# BoilDesc LLM service — models as tools, Gemini as the reasoner

This notebook turns the three trained models of the boiling-descriptor pipeline into a
**standalone local service** an LLM (Gemini, via its OpenAI-compatible API) drives through
tool calling:

```
 you (browser, natural language)
   │  "what does the wall region look like at 21 °C?"
   ▼
 Gemini classifies the question (7-way routing table in its context)
   │  render · forecast · conditions · reliability · sensitivity · nearest-real · refuse
   ▼
   • boiling_render   →  M4 (CNN generator) renders a frame AND M2 (retriever) returns the
                         nearest REAL simulated frame → two temperature images + merged table
   • list_conditions  →  what the 15 simulations contain
   • boiling_forecast →  the TCN rolls the 13 descriptors forward (short-term point,
                         long-term statistics, chaos-floor caveats)
   • reliability      →  answered directly from the model card (no tool call)
   ▼
 the answer merges both models as a range and shows each prediction
```

**v2 of this notebook** fixes the real-data failure of v1 (the M2 "self-retrieval
identity" assert): the retriever index of the real corpus contains exact duplicate
conditioning rows, so `argmin` legitimately returns an *earlier* duplicate — the
check now tests the true invariant (the nearest row is numerically identical, row 0
retrieves itself) instead of an index coincidence. All smoke tests and self-checks
were re-audited to the same standard: no hardcoded row indices or column names,
NaN/shape guards, a median-of-5 anchor check against the leaderboard reference,
an LLM-unreachable fallback in every test, and a **tail fast-path** so
"what happens next" forecasts stay fast on 4000-frame files (only onset-origin
questions stream the whole file, once, cached). Question routing is now an explicit
table shared by the system prompt, the offline placeholder LLM and the self-check.

**v3** fixes two more real-bundle incompatibilities found on the target corpus:
the M2 bundle's `twall` metadata carries **string labels** (`'Twall_10'`) rather than
numbers — all wall temperatures are now parsed robustly (`'Twall_10'` → 10, or the
`@10C` stem suffix, or the file's own attributes), and the wall-temperature cond
column is resolved **by the bundle** (`WALL_COL`, with `wallTemp`/`Twall` accepted as
ask aliases) instead of assuming the name; and the real cond matrix conditions on
**13 live columns** (f_hot_2–5 are dead and absent), so no code path indexes a cond
column that may not exist — round-trips are computed over the bundle's own live set.

**v7** three presentation adjustments: the two heat maps are back on **one
shared color scale** (vmin/vmax span both frames, the range printed on each
colorbar and caption — the same shade means the same temperature in both
panels); the suggestion buttons and the input field form a **sticky bottom
bar**, so the buttons ride on top of the field and stay visible on the empty
page, through answers, and while scrolling; and answers arrive as **prose
only** — the prompt forbids text tables and a conservative filter strips
markdown table lines, so nothing appears as a raw text table above the
"In short" summary box.

**v6** reworks how results are shown: the images are never cropped (explicit
figure padding keeps every title/colorbar inside the canvas — the clipped
forecast suptitle was the "cut-off image" — and the page shows images with
`object-fit: contain`), **each heat map gets its own color scale** with the
range printed on its colorbar and in the caption, the result cards lead with
stat chips (round-trip, retrieval gap, agreement, onset/burst), and every
answer ends with a deterministic **"In short"** box that explains the results
in plain sentences built from the tool JSONs.

**v5** fixes the stale-server trap behind the Windows "10048 — only one usage
of each socket address" error: a previous run's server kept the port and the
browser kept talking to that OLD instance (old page, no live thinking steps).
The launch cell now shuts down a server started earlier by the same kernel,
moves to the next free port (with a loud note) when 8000 is held by an older
run or another program, and only claims "up" when the NEW server answers the
probe — open the URL the last cell prints.

**v4** fixes the last two runtime issues: the LLM client is now the **same
provider-agnostic setup as the user's working standalone notebook** (`Bearer`
auth, minimal body, tools only when present, 4xx permanent with the server's
message quoted verbatim, `llm_ping()` connectivity check at cell 7 — no more
opaque "400 Bad Request after retries"), and the browser shows **live thinking
feedback**: while `/chat` runs, the page polls `/progress` and displays the
loop's current step ("thinking · step 3: calling boiling_render(...)") instead
of a silent wait.

**Everything is auto-discovered** under `DATA_DIR` (default `G:\BoilDesc`): the three
bundles (`M4_generator_bundle_*`, `M2_k1_retriever_bundle_*`, `FC_F2_TCN_bundle_*` — any
subfolder, newest timestamp wins) and every `.h5` simulation file. No paths are hardcoded.

**Requirements** (pip install once): `torch h5py numpy scipy pandas matplotlib fastapi
uvicorn requests pillow jupyter`. For the LLM: a Google AI Studio key in `.env` (see
below). Without a key the service still runs — the chat answers come from a scripted
placeholder loop (for testing), and `POST /tools/render` works without any LLM.

**How to run**: run all cells top to bottom. The last cell starts the service at
`http://localhost:8000` **in a background thread** — open it in the browser; the
kernel stays usable and the server stops when you restart the kernel. (Serving
runs in a thread because `uvicorn.run()` would call `asyncio.run()` inside
Jupyter's already-running event loop.) The outputs embedded in *this* copy were
produced by the built-in **fixture smoke test** (synthetic tiny data, because the machine
that generated it has no `G:\BoilDesc`); on your machine the same cells run against the
real bundles and print the real artifact table.

Config lives in `.env` next to this notebook (created if missing):

```ini
DATA_DIR     = G:/BoilDesc
LLM_BASE_URL = https://generativelanguage.googleapis.com/v1beta/openai/
LLM_API_KEY  =            # <- paste your Google AI Studio key here
LLM_MODEL    = gemini-3.1-flash-lite
PORT         = 8000
```


## 1 · Configuration and artifact auto-discovery

Reads `.env` (created from a template on first run — paste the Gemini key there),
then walks `DATA_DIR` recursively for the three bundles and every `.h5` simulation:
patterns `M4_generator_bundle_*`, `M2_k1_retriever_bundle_*`, `FC_*_bundle*`
(the TCN with or without a `.pt` extension; non-TCN bundles are skipped with a
message), newest filename stamp wins, every candidate validated by loading.
Nothing is hardcoded — move the files, add a re-trained bundle with a newer stamp,
and it is picked up automatically. If nothing is found the notebook switches to
**fixture mode** and CELL 3 builds a synthetic corpus with the identical schema so
the whole pipeline still runs end-to-end.

In [1]:
# CELL 1 · configuration, .env, and artifact auto-discovery
#
# Finds the three bundles + the .h5 corpus anywhere under DATA_DIR (glob patterns,
# newest timestamp wins, each candidate validated by loading). If nothing is found
# (e.g. a machine without G:\BoilDesc) the notebook switches to FIXTURE mode: CELL 3
# builds a tiny synthetic corpus + fake bundles with the exact same schema, so every
# downstream cell runs the identical code path. Set BOILDESC_FIXTURE=1 to force it.
import os, re, json, glob, math, time
import numpy as np

BASE = os.getcwd()
RUNTIME_DIR = os.path.join(BASE, 'boildesc_runtime')      # images, audit log, fixtures
os.makedirs(RUNTIME_DIR, exist_ok=True)

# ---- .env (never overrides real environment variables) -----------------------
_env_path = os.path.join(BASE, '.env')
if not os.path.exists(_env_path):
    with open(_env_path, 'w', encoding='utf-8') as f:
        f.write('# BoilDesc service configuration\n'
                'DATA_DIR=G:/BoilDesc\n'
                'LLM_BASE_URL=https://generativelanguage.googleapis.com/v1beta/openai/\n'
                'LLM_API_KEY=\n'
                'LLM_MODEL=gemini-3.1-flash-lite\n'
                'PORT=8000\n')
    print(f"[env] template written -> {_env_path}  (paste your API key there)")
for _ln in open(_env_path, encoding='utf-8').read().splitlines():
    _ln = _ln.strip()
    if not _ln or _ln.startswith('#') or '=' not in _ln:
        continue
    _k, _v = _ln.split('=', 1)
    os.environ.setdefault(_k.strip(), _v.strip().strip('"').strip("'"))

DATA_DIR     = os.environ.get('DATA_DIR', 'G:/BoilDesc')
LLM_BASE_URL = os.environ.get('LLM_BASE_URL',
                              'https://generativelanguage.googleapis.com/v1beta/openai/')
LLM_API_KEY  = os.environ.get('LLM_API_KEY', '').strip()
LLM_MODEL    = os.environ.get('LLM_MODEL', 'gemini-3.1-flash-lite')
PORT         = int(os.environ.get('PORT', '8000'))
_FIX_ENV     = os.environ.get('BOILDESC_FIXTURE', '').strip().lower() in ('1', 'true', 'yes')

def _stamp(path):
    """newest-artifact ordering: filename stamp yyyymmdd_hhmm, else mtime."""
    m = re.search(r'(\d{8}_\d{4})', os.path.basename(path))
    return m.group(1) if m else time.strftime('%Y%m%d_%H%M',
                                              time.localtime(os.path.getmtime(path)))

def _glob_all(root, patterns):
    out = []
    for pat in patterns:
        out += glob.glob(os.path.join(root, '**', pat), recursive=True)
    return sorted({os.path.normpath(p) for p in out
                   if not any(b in p for b in ('.venv', 'site-packages',
                                               '__pycache__', '.git'))})

def discover(root):
    """pattern search -> validated artifacts (dict). Missing items stay None."""
    import torch
    found = dict(m4=None, m2=None, fc=None, registry=None, h5=[])
    found['h5'] = _glob_all(root, ['*.h5', '*.hdf5'])
    for key, pats in (('m4', ['M4_generator_bundle*.pt', 'M4_generator_bundle*']),
                      ('m2', ['M2_k1_retriever_bundle*.pt',
                              'M2_k1_retriever_bundle*']),
                      ('fc', ['FC_*_bundle*.pt', 'FC_*_bundle?*'])):
        cands = sorted(_glob_all(root, pats), key=_stamp)
        for p in reversed(cands):                       # newest first
            try:
                b = torch.load(p, map_location='cpu', weights_only=False)
                want = {'m4': 'M4_generator', 'm2': 'M2_k1_retriever',
                        'fc': 'FC_forecaster'}[key]
                if isinstance(b, dict) and b.get('kind') == want:
                    if key == 'fc' and b.get('model_name') != 'F2 TCN':
                        print(f"[disc] skipping {os.path.basename(p)} "
                              f"(model_name={b.get('model_name')!r}, want 'F2 TCN')")
                        continue
                    found[key] = dict(path=p, bundle=b, stamp=_stamp(p))
                    break
            except Exception as e:
                print(f"[disc] {os.path.basename(p)} failed to load: {e}")
    regs = _glob_all(root, ['model_registry.json'])
    if regs:
        try:
            found['registry'] = dict(path=regs[-1],
                                     content=json.load(open(regs[-1], encoding='utf-8')))
        except Exception:
            pass
    return found

def report(art):
    rows = [('M4 generator bundle', art['m4']), ('M2 retriever bundle', art['m2']),
            ('FC TCN bundle', art['fc']),
            ('h5 simulations', art['h5'] and
             f"{len(art['h5'])} file(s), e.g. {os.path.basename(art['h5'][0])}")]
    for name, v in rows:
        if isinstance(v, dict):
            print(f"[disc] {name:22s} -> {v['path']}  (stamp {v['stamp']})")
        elif isinstance(v, str):
            print(f"[disc] {name:22s} -> {v}")
        else:
            print(f"[disc] {name:22s} -> MISSING")
    if art['registry']:
        n = sum(len(v) if isinstance(v, list) else 1
                for v in art['registry']['content'].values())
        print(f"[disc] registry           -> {art['registry']['path']} "
              f"({n} entries, informational)")

ART = discover(DATA_DIR)
FIXTURE = _FIX_ENV or not (ART['m4'] and ART['m2'] and ART['h5'])
if FIXTURE:
    FIX_DIR = os.path.join(RUNTIME_DIR, 'fixtures')
    print(f"[mode] FIXTURE — no usable artifacts under {DATA_DIR} (or BOILDESC_FIXTURE=1).")
    print(f"        CELL 3 will build a synthetic corpus + fake bundles under {FIX_DIR};")
    print("        every number this run prints is a synthetic placeholder.")
else:
    print(f"[mode] REAL DATA — artifacts under {DATA_DIR}")
    report(ART)
    if not ART['fc']:
        print("[mode] no FC_TCN bundle found -> boiling_forecast disabled (render-only)")

print(f"[cfg] DATA_DIR={DATA_DIR}  LLM={LLM_MODEL} "
      f"@ {LLM_BASE_URL.split('/')[2]}  "
      f"key={'set' if LLM_API_KEY else 'NOT set (offline placeholder loop)'}  port={PORT}")


[disc] fc_tcn_bundle_check.ipynb failed to load: invalid load key, '{'.
[mode] REAL DATA — artifacts under G:/BoilDesc
[disc] M4 generator bundle    -> G:\BoilDesc\src\models\M4_generator_bundle_f15files_20261106_0718.pt  (stamp 20261106_0718)
[disc] M2 retriever bundle    -> G:\BoilDesc\src\models\M2_k1_retriever_bundle_f15files_20261106_0718.pt  (stamp 20261106_0718)
[disc] FC TCN bundle          -> G:\BoilDesc\src\FC_F2_TCN_bundle_f15files_20261106_0927.pt  (stamp 20261106_0927)
[disc] h5 simulations         -> 15 file(s), e.g. Twall_100.hdf5
[disc] registry           -> G:\BoilDesc\src\new_predictions single bubble\model_registry.json (2 entries, informational)
[cfg] DATA_DIR=G:/BoilDesc  LLM=gemini-3.1-flash-lite @ generativelanguage.googleapis.com  key=set  port=8000


## 2 · Runtime ports 

The descriptor extractor, the h5 schema helpers, the M4 Generator (FiLM-conditioned
upsample+conv CNN) and the TCN architecture are ported **verbatim** from the two
training notebooks. This matters: the round-trip numbers are only meaningful against
the identical code that produced the bundles, so nothing here is re-derived. The
extractor state (thresholds, region geometry) is bound later from the M4 bundle's
own `cfg` — the training conventions travel with the weights.

In [2]:
# CELL 2 · runtime ports — verbatim from the training notebooks
#
# The descriptor extractor, the h5 schema helpers, the M4 Generator and the TCN
# architecture are copied as-is from M4_v3 / FC (only parameterized by bundle values
# instead of notebook globals). The round-trip numbers are only meaningful against
# the identical code that produced the bundles, so nothing is re-derived here.
import re, math
import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import ndimage

# ---- config defaults = the training CFG (real mode overwrites from the M4 bundle) ----
SVC_CFG = dict(
    KEY_DFUN='dfun', KEY_T='temperature',
    WTEMP_ATTRS=('wall_temperature', 'wallTemp', 'wall_temp'),
    WTEMP_FN_REGEX=r'twall[_-]?(\d+(?:\.\d+)?)',
    DT_ATTRS=('dt', 'time_step', 'deltaT'), DT_DEFAULT=0.1,
    N_REGIONS=5, REGION_AXIS='y', REGION_FLIP=False,
    TAU_DFUN=0.0, T_HOT_MODE='wall_minus_margin', T_HOT_MARGIN=5.0, T_HOT_ABS=65.0,
    T_MEAN_LIQUID=False, MIN_BUB_PIX=6,
    MIN_LAT=8, LAT_CH=96, CHS_EXTRA=[128, 96, 64, 48], Z_DIM=0,
    CLIP_B=4.0, CLIP_INF=6.0, BOILING_TWALL_MIN=70.0,
)

# ============================== h5 schema helpers (M4_v3 CELL 1) ===============
def as_float(v):
    """numeric scalar or None (strings / arrays / compound junk -> None)."""
    try:
        a = np.asarray(v)
        if a.dtype.kind in 'US':
            return None
        a = np.ravel(a)
        return float(a[0]) if a.size == 1 else None
    except Exception:
        return None

def scalar_candidates(f):
    """numeric scalars from attrs + ()/(1,) root datasets."""
    out = {}
    for k, v in f.attrs.items():
        x = as_float(v)
        if x is not None:
            out[k] = x
    for k in f.keys():
        try:
            d = f[k]
            if isinstance(d, h5py.Dataset) and d.ndim <= 1 and d.size <= 1:
                x = as_float(d[()])
                if x is not None and k not in out:
                    out[k] = x
        except Exception:
            pass
    return out

def find_dfun_key(f):
    k = SVC_CFG['KEY_DFUN']
    if k in f.keys() and f[k].ndim == 3:
        return k
    for k in f.keys():
        if isinstance(f[k], h5py.Dataset) and f[k].ndim == 3 and 'dfun' in k.lower():
            return k
    return None

def find_T_key(f):
    cand = [k for k in f.keys()
            if isinstance(f[k], h5py.Dataset) and f[k].ndim == 3
            and 'temp' in k.lower()]
    cand.sort(key=lambda k: (k.lower() != 'temperature',
                             any(w in k.lower() for w in ('wall', 'sat', 'subcool'))))
    return cand[0] if cand else None

def _wt_num(v):
    """wall temperature as a float from anything: a number, a numeric string,
    a 'Twall_10'-style label (trailing number) or an '@10C' stem suffix."""
    x = as_float(v)
    if x is not None:
        return x
    s = str(v).strip()
    m = re.search(r'(-?\d+(?:\.\d+)?)\s*$', s)
    if not m:
        m = re.search(r'@(-?\d+(?:\.\d+)?)\s*c?\b', s, re.I)
    if not m:
        m = re.search(r'(-?\d+(?:\.\d+)?)\s*°?c\b', s, re.I)
    return float(m.group(1)) if m else float('nan')

def _wt_stem(stem):
    """wall temperature from a simulation stem only ('f8106@10C' -> 10; else nan).
    Deliberately narrower than _wt_num: a plain trailing number in a file id is
    not a temperature."""
    s = str(stem)
    for rx in (r'@(-?\d+(?:\.\d+)?)\s*c?\b', r'(-?\d+(?:\.\d+)?)\s*°?c\b'):
        m = re.search(rx, s, re.I)
        if m:
            return float(m.group(1))
    return float('nan')

def walltemp_of(path, sc, raw_attrs=None):
    """wall temperature: named attr -> any wall/tw attr (numeric value OR a
    string label like 'Twall_10') -> file-name regex."""
    for name in SVC_CFG['WTEMP_ATTRS']:
        for k in sc:
            if k.lower() == name.lower():
                return sc[k], k
    for k, v in {**(raw_attrs or {}), **sc}.items():
        if re.search(r'wall|^tw', k.lower()):
            x = _wt_num(v)
            if np.isfinite(x):
                return x, k
    for rx in (SVC_CFG['WTEMP_FN_REGEX'], r'@(-?\d+(?:\.\d+)?)C'):
        m = re.search(rx, os.path.basename(path), re.I)
        if m:
            return float(m.group(1)), 'filename'
    return float('nan'), None

def centre_crop(a, H_, W_):
    h, w = a.shape[-2:]
    if (h, w) == (H_, W_):
        return a
    y0, x0 = (h - H_) // 2, (w - W_) // 2
    return a[..., y0:y0 + H_, x0:x0 + W_]

import os  # noqa: E402 (kept next to its first use, as in the notebooks)

# ============================== descriptor extractor (M4_v3 CELL 1) ===========
_H5 = {}                       # lazy h5py handle pool: path -> (File, kd, kt)
_FRAME_KEYS = {}               # stem -> (kd, kt) cache

def h5_open(path):
    if path not in _H5:
        f = h5py.File(path, 'r')
        _H5[path] = (f, find_dfun_key(f), find_T_key(f))
    return _H5[path]

def frame_of(path, pos):
    """one (dfun, T) frame, cropped to the common geometry."""
    f, kd, kt = h5_open(path)
    d = np.asarray(f[kd][pos], dtype=np.float32)
    T = np.asarray(f[kt][pos], dtype=np.float32)
    return centre_crop(d, H, W), centre_crop(T, H, W)

def file_meta(path):
    """(n_frames, wallTemp, dt) for a discovered simulation."""
    f, kd, kt = h5_open(path)
    n = f[kd].shape[0]
    sc = scalar_candidates(f)
    wt, _src = walltemp_of(path, sc, raw_attrs=dict(f.attrs.items()))
    dt = next((sc[a] for a in SVC_CFG['DT_ATTRS'] if a in sc), SVC_CFG['DT_DEFAULT'])
    return int(n), _wt_num(wt), _wt_num(dt)

def build_regions(H_, W_):
    """equal bands along REGION_AXIS; region 1 = first band along that axis."""
    n = SVC_CFG['N_REGIONS']
    if SVC_CFG['REGION_AXIS'] == 'y':
        e = np.linspace(0, H_, n + 1).astype(int)
        sl = [(slice(e[r], e[r + 1]), slice(0, W_)) for r in range(n)]
    else:
        e = np.linspace(0, W_, n + 1).astype(int)
        sl = [(slice(0, H_), slice(e[r], e[r + 1])) for r in range(n)]
    return sl[::-1] if SVC_CFG['REGION_FLIP'] else sl

REG_SLICES = None              # set by init_extractor below

def extract_theta(dfun, T, t_hot):
    """THE descriptor extractor — one code path for real and generated frames.
    That single fact is what makes the round-trip well-defined."""
    mask = dfun > SVC_CFG['TAU_DFUN']
    out = {}
    for r, (sy, sx) in enumerate(REG_SLICES, 1):
        m_r = mask[sy, sx]
        T_r = T[sy, sx]
        out[f'alpha_{r}'] = float(m_r.mean())
        if SVC_CFG['T_MEAN_LIQUID']:
            w = (~m_r).astype(np.float32)
            out[f'Tmean_{r}'] = float((T_r * w).sum() / max(w.sum(), 1.0))
        else:
            out[f'Tmean_{r}'] = float(T_r.mean())
        out[f'f_hot_{r}'] = float((T_r > t_hot).mean())
    lab, nb = ndimage.label(mask)
    if nb:
        areas = np.bincount(lab.ravel())[1:]
        areas = areas[areas >= SVC_CFG['MIN_BUB_PIX']]
        out['nbub'] = int(len(areas))
        if len(areas):
            d_eq = 2.0 * np.sqrt(areas / np.pi)
            out['D32'] = float((d_eq ** 3).sum() / max((d_eq ** 2).sum(), 1e-9))
        else:
            out['D32'] = 0.0
    else:
        out['nbub'] = 0
        out['D32'] = 0.0
    return out

def theta_of_frames(dNp, TNp, thot):
    """numpy round-trip: run generated (or real) frames through extract_theta."""
    return np.stack([[extract_theta(dNp[i], TNp[i], float(thot[i]))[c]
                      for c in FRAME_COLS] for i in range(len(dNp))],
                    axis=0).astype(np.float32)

def t_hot_of_walltemp(wt):
    """hot threshold per condition (travels with the wall temperature by default)."""
    if SVC_CFG['T_HOT_MODE'] == 'absolute':
        return float(SVC_CFG['T_HOT_ABS'])
    if np.isfinite(wt):
        return float(wt - SVC_CFG['T_HOT_MARGIN'])
    return None

def init_extractor(cfg, H_, W_, frame_cols):
    """bind the extractor state (called once the bundle geometry is known)."""
    global H, W, REG_SLICES, FRAME_COLS
    SVC_CFG.update({k: v for k, v in cfg.items() if k in SVC_CFG})
    H, W = int(H_), int(W_)
    REG_SLICES = build_regions(H, W)
    FRAME_COLS = list(frame_cols)
    return REG_SLICES

# ============================== M4 Generator (M4_v3 CELL 3) ===================
class FiLM(nn.Module):
    """per-channel affine modulation driven by the descriptor vector — the
    conditioning mechanism that makes theta a *control* input, not a lookup key."""
    def __init__(self, cond_dim, ch):
        super().__init__()
        self.net = nn.Linear(cond_dim, 2 * ch)
    def forward(self, x, cond):
        g, b = self.net(cond).chunk(2, dim=1)
        return x * (1 + g[:, :, None, None]) + b[:, :, None, None]

class GenBlock(nn.Module):
    """one 2x stage. Upsample+Conv instead of ConvTranspose2d (no checkerboard)."""
    def __init__(self, ch_in, ch_out, cond_dim):
        super().__init__()
        self.up = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(ch_in, ch_out, 3, padding=1))
        self.film1 = FiLM(cond_dim, ch_out)
        self.conv = nn.Conv2d(ch_out, ch_out, 3, padding=1)
        self.film2 = FiLM(cond_dim, ch_out)
        self.act = nn.LeakyReLU(0.2)
    def forward(self, x, cond):
        x = self.act(self.film1(self.up(x), cond))
        return self.act(self.film2(self.conv(x), cond))

class Generator(nn.Module):
    """theta (z-scored) -> (2, H, W) normalized frame."""
    def __init__(self, cond_dim, lat_ch, chs, out_ch, lat_hw):
        super().__init__()
        self.lat_hw = tuple(lat_hw)
        self.fc = nn.Linear(cond_dim, lat_ch * lat_hw[0] * lat_hw[1])
        chans = [lat_ch] + list(chs)
        self.blocks = nn.ModuleList(
            GenBlock(chans[i], chans[i + 1], cond_dim) for i in range(len(chans) - 1))
        self.head = nn.Conv2d(chans[-1], out_ch, 3, padding=1)   # linear head

    def forward(self, cond):
        B = cond.size(0)
        x = self.fc(cond).view(B, -1, self.lat_hw[0], self.lat_hw[1])
        for blk in self.blocks:
            x = blk(x, cond)
        return self.head(x)

def build_generator(cfg, cond_dim, H_, W_):
    """resolution ladder derived from the frame size — pixel-exact output."""
    NST = max(1, int(math.log2(min(H_, W_) / cfg['MIN_LAT'])))
    LAT_H, LAT_W = H_ // 2 ** NST, W_ // 2 ** NST
    chs = list(cfg['CHS_EXTRA'])
    chs = chs[:NST] if len(chs) >= NST else chs + [16] * (NST - len(chs))
    cond = cond_dim + cfg.get('Z_DIM', 0)
    return Generator(cond, cfg['LAT_CH'], chs, 2, (LAT_H, LAT_W)), (LAT_H, LAT_W), NST

# ============================== TCN (FC CELL 4) ===============================
class CausalConv1d(nn.Module):
    """dilated causal conv with a (non-gated) residual block, dropout inside."""
    def __init__(self, ch, dil, k, drop):
        super().__init__()
        pad = (k - 1) * dil
        self.conv = nn.Conv1d(ch, ch, k, dilation=dil, padding=pad)
        self.norm = nn.GroupNorm(8, ch)
        self.drop = nn.Dropout(drop)
    def forward(self, x):                                  # x: (B, ch, T)
        y = self.drop(F.relu(self.norm(self.conv(x))[:, :, : x.size(2)]))
        return x + y

class TCNNet(nn.Module):
    """dilated causal TCN over the (z | case) input stream -> dz at the last step."""
    def __init__(self, in_dim, L, ch, k, blocks, drop):
        super().__init__()
        self.inp = nn.Conv1d(in_dim, ch, 1)
        self.blocks = nn.ModuleList(
            [CausalConv1d(ch, 2 ** b, k, drop) for b in range(blocks)])
        self.head = nn.Conv1d(ch, L, 1)

    def forward(self, x):                                  # (B, WIN, in_dim) -> (B, L)
        h = self.inp(x.transpose(1, 2))                    # (B, ch, WIN)
        for b in self.blocks:
            h = b(h)
        return self.head(h[:, :, -1:]).squeeze(-1)

print('[ok] CELL 2 done — extractor, h5 helpers, Generator and TCN classes ported')


[ok] CELL 2 done — extractor, h5 helpers, Generator and TCN classes ported


## 3 · Load M4 + M2  

Real mode: rebuild the Generator from the bundle's `cfg` + `state_dict`, take the
retriever's conditioning matrix and provenance, and smoke-test both. The smoke
tests check **invariants, not index coincidences**: any stored cond row must
retrieve a *numerically identical* row (the real index contains exact duplicate
rows — quiescent frames of one simulation can share a descriptor vector to
float32 — and `argmin` then legitimately returns the first of them); row 0, which
nothing precedes, must retrieve itself; the generator output must be finite and of
the exact geometry; and a retrieved row's provenance must resolve to a readable
real frame. Shape and NaN guards repair a damaged index with a loud note, the
anchor round-trips report a median over five spread rows against the leaderboard
reference, and stems are matched against the discovered files with a containment
fallback. The wall-temperature cond column is resolved from the bundle (`WALL_COL`)
— its metadata can be stringly (`'Twall_10'`) and the live set is 13 of the 17
frame columns, so nothing assumes the column name or a full cond coverage.
Fixture mode: a tiny two-simulation corpus and fake bundles built *with
the real classes of CELL 2*, then the identical loading path.

In [3]:
# CELL 3 · fixture corpus + load M4 & M2
#
# Real mode: reads the two bundles discovered in CELL 1. Fixture mode: builds a tiny
# synthetic corpus + fake bundles FIRST (same schema, same classes as CELL 2), then
# re-runs discovery so the loading code below is byte-identical in both modes.
#
# v2: the smoke tests check INVARIANTS, not index coincidences. The retriever index
# of the real 15-file corpus contains exact duplicate conditioning rows (quiescent
# frames of the same simulation can share a descriptor vector to float32), so
# np.argmin legitimately returns an EARLIER duplicate — "row 17 retrieves row 17"
# is not a valid test. Valid tests: (a) the nearest row is numerically identical
# to the query row, (b) row 0 (nothing earlier) retrieves itself, (c) the
# generator output has the exact geometry and is finite, (d) a retrieved row's
# provenance resolves to a readable real frame.
import json
import numpy as np
import torch

FRAME_COLS = ([f'alpha_{r}' for r in range(1, SVC_CFG['N_REGIONS'] + 1)]
              + [f'Tmean_{r}' for r in range(1, SVC_CFG['N_REGIONS'] + 1)]
              + [f'f_hot_{r}' for r in range(1, SVC_CFG['N_REGIONS'] + 1)]
              + ['D32', 'nbub'])

if FIXTURE:
    # ---------------- tiny synthetic corpus (2 sims, 48 frames each) -------------
    FIX_DIR = os.path.join(RUNTIME_DIR, 'fixtures')
    os.makedirs(os.path.join(FIX_DIR, 'models'), exist_ok=True)
    rng = np.random.default_rng(7)
    H_FIX, W_FIX = 288, 192
    FIX_CFG = dict(SVC_CFG)          # extractor conventions identical to training
    FIX_CFG.update(MIN_LAT=12, LAT_CH=8, CHS_EXTRA=[16, 12, 12, 8])
    init_extractor(FIX_CFG, H_FIX, W_FIX, FRAME_COLS)

    def _fake_file(path, wt, n, nb_fn):
        with h5py.File(path, 'w') as f:
            d = np.zeros((n, H_FIX, W_FIX), np.float32)
            T = np.zeros((n, H_FIX, W_FIX), np.float32)
            yy = np.linspace(0, 1, H_FIX)[:, None]
            for t in range(n):
                nb = max(0, int(round(nb_fn(t))))
                fr = np.zeros((H_FIX, W_FIX), np.float32)
                for _ in range(nb):
                    cy, cx = rng.uniform(0, H_FIX), rng.uniform(0, W_FIX)
                    s = rng.uniform(4, 11)
                    y0, y1 = max(0, int(cy - 3 * s)), min(H_FIX, int(cy + 3 * s))
                    x0, x1 = max(0, int(cx - 3 * s)), min(W_FIX, int(cx + 3 * s))
                    gy = (np.arange(y0, y1) - cy)[:, None]
                    gx = (np.arange(x0, x1) - cx)[None, :]
                    fr[y0:y1, x0:x1] += 2.0 * np.exp(-(gy ** 2 + gx ** 2) / (2 * s * s))
                T[t] = (wt - 4.0 - 8.0 * yy
                        + math.sin(t / 5.0) * np.clip(1.0 - 5.5 * yy, 0.0, 1.0)
                        + 0.10 * fr + rng.normal(0, .1, (H_FIX, W_FIX))).astype(np.float32)
                d[t] = (fr - 0.05 + rng.normal(0, .02, (H_FIX, W_FIX))).astype(np.float32)
            f.create_dataset('dfun', data=d)
            f.create_dataset('temperature', data=T)
            f.attrs['wallTemp'] = float(wt)
            f.attrs['dt'] = 0.1

    stems_fix = ['f101@20C', 'f102@85C']
    for stem, wt in zip(stems_fix, (20.0, 85.0)):
        _fake_file(os.path.join(FIX_DIR, stem + '.h5'), wt, 48,
                   (lambda t: 0 if t < 6 else 3 + rng.normal(0, .7)) if wt < 60
                   else lambda t: 2 + 8 * (np.sin(t / 5.0) > .6) + rng.normal(0, .7))
    import pandas as pd
    rows = []
    for stem, wt in zip(stems_fix, (20.0, 85.0)):
        with h5py.File(os.path.join(FIX_DIR, stem + '.h5'), 'r') as f:
            d = np.asarray(f['dfun'], np.float32)
            T = np.asarray(f['temperature'], np.float32)
        thot = t_hot_of_walltemp(wt)
        for pos in range(d.shape[0]):
            rows.append(dict(file=stem, pos=pos, t=pos * 0.1, wallTemp=wt,
                             **extract_theta(d[pos], T[pos], thot)))
    ROWS = pd.DataFrame(rows)
    LIVE_FIX = [c for c in FRAME_COLS if ROWS[c].std() > 1e-6]
    COND_FIX = LIVE_FIX + ['wallTemp']
    TH_MU_FIX = {c: float(ROWS[c].mean()) for c in FRAME_COLS + ['wallTemp']}
    TH_SD_FIX = {c: max(float(ROWS[c].std()), 1e-9) for c in FRAME_COLS + ['wallTemp']}
    ZC_FIX = ((ROWS[COND_FIX].values - np.array([TH_MU_FIX[c] for c in COND_FIX]))
              / np.array([TH_SD_FIX[c] for c in COND_FIX])).astype(np.float32)
    ds, Ts = [], []
    for stem in stems_fix:
        with h5py.File(os.path.join(FIX_DIR, stem + '.h5'), 'r') as f:
            ds.append(np.asarray(f['dfun'], np.float32).ravel())
            Ts.append(np.asarray(f['temperature'], np.float32).ravel())
    _d = np.concatenate(ds)[::7]
    _t = np.concatenate(Ts)[::7]
    NORM_FIX = dict(mu_d=float(_d.mean()), sd_d=max(float(_d.std()), 1e-6),
                    mu_T=float(_t.mean()), sd_T=max(float(_t.std()), 1e-6))
    # ---- fake bundles built with the REAL classes of CELL 2 -------------------
    torch.manual_seed(11)
    gnet, _, _nst = build_generator(FIX_CFG, len(COND_FIX), H_FIX, W_FIX)
    _p = os.path.join(FIX_DIR, 'models', 'M4_generator_bundle_f2files_20261126_0001.pt')
    torch.save(dict(kind='M4_generator', created='20261126_0001', n_files=2,
                    files=stems_fix, twall=[20.0, 85.0],
                    state_dict={k: v.clone() for k, v in gnet.state_dict().items()},
                    best_ep=1, cfg=json.dumps(FIX_CFG), cond_cols=COND_FIX,
                    frame_cols=FRAME_COLS,
                    col2cond={c: j for j, c in enumerate(COND_FIX)},
                    th_mu=TH_MU_FIX, th_sd=TH_SD_FIX, norm=NORM_FIX, CAL_ref=None,
                    CAL_applied_to_predictions=False, howto='fixture'), _p)
    torch.manual_seed(12)
    tnet = TCNNet(len(LIVE_FIX) + 1, len(LIVE_FIX), 16, 3, 3, 0.1)
    _f = os.path.join(FIX_DIR, 'models', 'FC_F2_TCN_bundle_f2files_20261126_0001.pt')
    torch.save(dict(kind='FC_forecaster', created='20261126_0001', model_name='F2 TCN',
                    win=8, h_max=80, live_cols=LIVE_FIX, case_cols=['wallTemp'],
                    th_mu=TH_MU_FIX, th_sd=TH_SD_FIX, frame_cols=FRAME_COLS,
                    z_lo=[-2.5] * len(LIVE_FIX), z_hi=[2.5] * len(LIVE_FIX),
                    onset_th={'F2 TCN': 0.5},
                    cfg=json.dumps(dict(TCN_CH=16, TCN_KERNEL=3, TCN_BLOCKS=3,
                                        DROPOUT=0.1, CLAMP_ROLLOUT=True, WIN=8, H_MAX=80)),
                    lead=[], onset=[], events=[], howto='fixture',
                    state_dict={k: v.clone() for k, v in tnet.state_dict().items()}), _f)
    _m = os.path.join(FIX_DIR, 'models', 'M2_k1_retriever_bundle_f2files_20261126_0001.pt')
    torch.save(dict(kind='M2_k1_retriever', created='20261126_0001', n_files=2,
                    files=stems_fix, twall=[20.0, 85.0],
                    cond=torch.from_numpy(ZC_FIX.astype(np.float32)),
                    cond_cols=COND_FIX, i_wt=COND_FIX.index('wallTemp'),
                    row_index=torch.from_numpy(np.arange(len(ROWS))),
                    row_file=list(ROWS['file'].values),
                    row_pos=torch.from_numpy(ROWS['pos'].values.astype(np.int64)),
                    frame_hw=(H_FIX, W_FIX), wall_side=None, gas_sign_of_dfun=1,
                    CAL_ref=None, CAL_applied_to_predictions=False, howto='fixture'), _m)
    with open(os.path.join(FIX_DIR, 'models', 'model_registry.json'), 'w') as fh:
        json.dump({'20261126_0001': [_m, _p]}, fh)
    del gnet, tnet
    ART = discover(FIX_DIR)
    report(ART)

# ============================== load M4 + M2 (identical in both modes) =========
B4 = ART['m4']['bundle']
B2 = ART['m2']['bundle']
M4_PATH, M2_PATH = ART['m4']['path'], ART['m2']['path']
CFG4 = {k: v for k, v in json.loads(B4['cfg']).items()}
H, W = (int(v) for v in B2['frame_hw'])
init_extractor(CFG4, H, W, B4['frame_cols'])
COND_COLS = list(B4['cond_cols'])
LIVE_COLS = [c for c in COND_COLS if c in FRAME_COLS]      # cond minus case scalars
# ---- the wall-temperature column, resolved generically: the real bundle's
# cond column need not be named 'wallTemp' (its twall metadata even carries
# string labels like 'Twall_10'), so every consumer uses WALL_COL instead ----
I_WT = B2.get('i_wt')
if I_WT is None or not (0 <= int(I_WT) < len(COND_COLS)):
    I_WT = next((j for j, c in enumerate(COND_COLS)
                 if re.search(r'wall|temp', c, re.I)), None)
if I_WT is None:
    I_WT = next((j for j, c in enumerate(COND_COLS) if c not in FRAME_COLS), None)
WALL_COL = COND_COLS[I_WT] if I_WT is not None else 'wallTemp'
TH_MU = {k: float(v) for k, v in B4['th_mu'].items()}
TH_SD = {k: float(v) for k, v in B4['th_sd'].items()}
NORM = {k: float(v) for k, v in B4['norm'].items()}

assert B4.get('state_dict'), \
    'M4 bundle carries no state_dict (provenance-only save) - re-export the weights'
M4NET, LAT_HW, NST = build_generator(CFG4, len(COND_COLS), H, W)
M4NET.load_state_dict(B4['state_dict'])
M4NET.eval()
_npar = sum(p.numel() for p in M4NET.parameters())

M2COND = (B2['cond'].numpy() if torch.is_tensor(B2['cond'])
          else np.asarray(B2['cond'])).astype(np.float32)
ROW_FILE = [str(s) for s in B2['row_file']]
ROW_POS = (B2['row_pos'].numpy() if torch.is_tensor(B2['row_pos'])
           else np.asarray(B2['row_pos'])).astype(np.int64)
# ---- index sanity (shape consistency + finite rows), repair with a loud note --
assert M2COND.ndim == 2, f'M2 cond is not a row matrix: shape {M2COND.shape}'
assert M2COND.shape[0] == len(ROW_FILE) == len(ROW_POS), \
    (f'M2 index inconsistent: cond {M2COND.shape[0]} rows vs {len(ROW_FILE)} '
     f'row_file vs {len(ROW_POS)} row_pos')
assert M2COND.shape[1] == len(COND_COLS), \
    f'M2 cond width {M2COND.shape[1]} != len(cond_cols) {len(COND_COLS)}'
_bad = ~np.isfinite(M2COND).all(axis=1)
if _bad.any():
    _keep = ~_bad
    M2COND = M2COND[_keep]
    ROW_FILE = [s for s, k in zip(ROW_FILE, _keep) if k]
    ROW_POS = ROW_POS[_keep]
    print(f'[m2] WARNING: dropped {_bad.sum()} non-finite cond rows from the '
          f'retriever index ({M2COND.shape[0]} kept)')

# stem -> h5 path, with a containment fallback (bundle stems vs file names)
STEM2PATH = {os.path.splitext(os.path.basename(p))[0]: p for p in ART['h5']}
for s in {f for f in ROW_FILE} - set(STEM2PATH):
    for stem, path in list(STEM2PATH.items()):
        if s and (s in stem or stem in s):
            STEM2PATH[s] = path
            break
_missing = sorted({s for s in ROW_FILE} - set(STEM2PATH))
if _missing:
    print(f"[m2] WARNING: {len(_missing)} retriever stems have no .h5 on disk "
          f"(e.g. {_missing[:3]}) - their frames cannot be materialised")

CAL_AB = None
if B4.get('CAL_ref') is not None:
    _cr = B4['CAL_ref']
    a, b = _cr if isinstance(_cr, (tuple, list)) and len(_cr) == 2 else (None, None)
    if a is not None:
        CAL_AB = ((a.numpy() if torch.is_tensor(a) else np.asarray(a)).astype(np.float64),
                  (b.numpy() if torch.is_tensor(b) else np.asarray(b)).astype(np.float64))
        if CAL_AB[0].shape[0] != len(FRAME_COLS):
            print(f'[m4] WARNING: CAL_ref has {CAL_AB[0].shape[0]} coefficients but '
                  f'frame_cols has {len(FRAME_COLS)} - ignoring CAL')
            CAL_AB = None

def apply_cal(theta_hat):
    """reporting-layer affine (never touches frames); identity when CAL_ref is None"""
    if CAL_AB is None:
        return np.asarray(theta_hat)
    return np.asarray(theta_hat) * CAL_AB[0] + CAL_AB[1]

def z_of_phys(phys_row):
    """{col: value} (physical) -> z vector in COND_COLS order."""
    return np.array([(phys_row.get(c, TH_MU[c]) - TH_MU[c]) / TH_SD[c]
                     for c in COND_COLS], np.float32)

def phys_of_z(zrow):
    return {c: float(zrow[j] * TH_SD[c] + TH_MU[c]) for j, c in enumerate(COND_COLS)}

def t_hot_of_cond_row(zrow):
    """hot threshold from a conditioning row (z space), training convention."""
    if SVC_CFG['T_HOT_MODE'] == 'absolute':
        return float(SVC_CFG['T_HOT_ABS'])
    if I_WT is None:                       # no wall column found anywhere
        _wts = [_wt_num(w) for w in B2['twall']]
        _wts = [w for w in _wts if np.isfinite(w)]
        return float(np.mean(_wts or [20.0])) - SVC_CFG['T_HOT_MARGIN']
    wt = float(zrow[I_WT] * TH_SD[COND_COLS[I_WT]] + TH_MU[COND_COLS[I_WT]])
    return float(wt - SVC_CFG['T_HOT_MARGIN'])

@torch.no_grad()
def generate(asks_z):
    """(B, cond_z) -> physical (dfun, T) frames; hard clamp at CLIP_INF sigma."""
    z = M4NET(torch.as_tensor(np.asarray(asks_z), dtype=torch.float32))
    z = z.clamp(-CFG4['CLIP_INF'], CFG4['CLIP_INF'])
    d = z[:, 0].numpy() * NORM['sd_d'] + NORM['mu_d']
    T = z[:, 1].numpy() * NORM['sd_T'] + NORM['mu_T']
    return (d.astype(np.float32), T.astype(np.float32), z.numpy())

def retrieve(q):
    """k=1 retrieval in cond z-space -> (k, distance, stem, pos)."""
    q = np.asarray(q, np.float32)
    dist = np.linalg.norm(M2COND - q[None], axis=1)
    k = int(np.argmin(dist))
    return k, float(dist[k]), ROW_FILE[k], int(ROW_POS[k])

# ---- smoke tests: invariants, not index coincidences --------------------------
# (a) any stored row retrieves a row numerically identical to itself (duplicates
#     may tie - argmin then returns the FIRST of them, which is still correct)
_probe = sorted({0, 17 % len(M2COND), len(M2COND) // 2, len(M2COND) - 1})
for j in _probe:
    k, d, s, p = retrieve(M2COND[j])
    _gap = float(np.linalg.norm(M2COND[k] - M2COND[j]))
    assert _gap < 1e-5, (f'M2 retrieval is inconsistent: query row {j} matched row '
                         f'{k} at residual gap {_gap:.2e} (not the same vector)')
# (b) row 0 has no earlier row, so it must retrieve itself exactly
_k0, _d0, _s0, _p0 = retrieve(M2COND[0])
assert _k0 == 0 and _d0 < 1e-5, f'M2 row 0 did not self-retrieve (k={_k0}, d={_d0})'
# (c) the generator produces finite frames of the exact geometry
_d1, _T1, _z1 = generate(M2COND[0:1])
assert _d1.shape == (1, H, W) and _T1.shape == (1, H, W), \
    f'M4 output shape {_d1.shape} != (1, {H}, {W})'
assert np.isfinite(_d1).all() and np.isfinite(_T1).all(), 'M4 output not finite'
# (d) a retrieved row's provenance resolves to a readable real frame
if _s0 in STEM2PATH:
    _dr, _Tr = frame_of(STEM2PATH[_s0], _p0)
    assert _dr.shape == (H, W) and np.isfinite(_dr).all() and np.isfinite(_Tr).all(), \
        f'real frame readback failed for {_s0}@{_p0}'
    print(f'[m2] frame readback: {_s0} pos {_p0} -> {_dr.shape} frame, finite PASS')
# (e) anchor round-trips: median over 5 spread rows (a single anchor can be a
#     blow-up frame - the leaderboard spreads over the whole test set)
if not FIXTURE:
    _idx = np.linspace(0, len(M2COND) - 1, 5).astype(int)
    _live = np.array([c in set(LIVE_COLS) for c in FRAME_COLS])
    _sd = np.array([TH_SD[c] for c in FRAME_COLS])
    _rts = []
    for j in _idx:
        d, T, _ = generate(M2COND[j:j + 1])
        th = apply_cal(theta_of_frames(d, T, [t_hot_of_cond_row(M2COND[j])]))[0]
        # .get: FRAME_COLS can contain columns the cond matrix never had (the
        # real live set is 13 of 17 - f_hot_2..5 are dead, not conditioned on)
        _ask = np.array([phys_of_z(M2COND[j]).get(c, np.nan) for c in FRAME_COLS],
                        np.float64)
        _rts.append(100 * float(np.mean(np.abs(th[_live] - _ask[_live]) / _sd[_live])))
    print(f'[m4] anchor round-trips at rows {_idx.tolist()}: '
          + ', '.join(f'{r:.1f}%' for r in _rts))
    print(f'[m4] median {float(np.median(_rts)):.1f}% '
          f'(leaderboard test reference: mean 11.7%, median 10.9%)')
print(f'[m4] Generator rebuilt: {LAT_HW[0]}x{LAT_HW[1]} latent, NST={NST}, '
      f'{_npar / 1e6:.2f} M params, loaded from {os.path.basename(M4_PATH)}')
print(f'[m2] retriever: {M2COND.shape[0]} rows x {M2COND.shape[1]} cols, '
      f'frames from {len(set(ROW_FILE))} file(s), '
      f'self-retrieval invariant PASS (rows {_probe} + row 0 exact)')
print('[ok] CELL 3 done — M4 + M2 live in the runtime')


[m2] frame readback: Twall_107 pos 0 -> (288, 192) frame, finite PASS
[m4] anchor round-trips at rows [0, 4001, 8003, 12005, 16007]: 7.6%, 7.6%, 6.5%, 6.6%, 13.9%
[m4] median 7.6% (leaderboard test reference: mean 11.7%, median 10.9%)
[m4] Generator rebuilt: 18x12 latent, NST=4, 0.92 M params, loaded from M4_generator_bundle_f15files_20261106_0718.pt
[m2] retriever: 16008 rows x 14 cols, frames from 8 file(s), self-retrieval invariant PASS (rows [0, 17, 8004, 16007] + row 0 exact)
[ok] CELL 3 done — M4 + M2 live in the runtime


## 4 · The TCN forecaster

The forecast bundle is rebuilt the way the FC notebook's own reload test does it:
`state_dict` → step function → autoregressive `rollout()` (the model's own
predictions fed back, clamped to the training range, clamps counted). Forecast
origins come from a real window of a chosen simulation — the θ time series is
streamed from the `.h5` file with the same extractor and cached (LRU, three
files). **v2 fast path**: `origin='end'` needs only the last WIN frames, so
"what happens next" answers in seconds even on 4000-frame files; only
`origin='onset'` (the window just before the first bubble — the design-relevant
question) streams the whole file, with progress printed, once, cached.

In [4]:
# CELL 4 · the TCN forecaster — bundle -> step function -> autoregressive rollout
#
# Ported from the FC notebook: the step function is rebuilt from the bundle's
# state_dict (the notebook's own rebuild path), rollout() is verbatim (feed the
# model its own predictions, clamp to the training range, count the clamps).
#
# v2: theta extraction has a TAIL fast path — origin='end' only needs the last
# WIN frames, so "what happens next" answers in seconds even for 4000-frame
# files; only origin='onset' (searching for the first bubble) streams the whole
# file, with progress printed, once, cached. Smoke tests use no hardcoded
# column names (the live set comes from the bundle).
import pandas as pd

FC_AVAILABLE = ART['fc'] is not None
if FC_AVAILABLE:
    BFC = ART['fc']['bundle']
    FCFG = json.loads(BFC['cfg'])
    LIVE_FC = list(BFC['live_cols'])
    CASE_FC = list(BFC['case_cols'])
    WIN = int(BFC['win'])
    H_MAX_FC = int(BFC['h_max'])
    TH_MU_FC = {k: float(v) for k, v in BFC['th_mu'].items()}
    TH_SD_FC = {k: float(v) for k, v in BFC['th_sd'].items()}
    Z_LO = np.asarray(BFC['z_lo'], np.float32)
    Z_HI = np.asarray(BFC['z_hi'], np.float32)
    ONSET_TH = float(BFC.get('onset_th', {}).get('F2 TCN', 0.5))
    TCN = TCNNet(len(LIVE_FC) + len(CASE_FC), len(LIVE_FC),
                 FCFG['TCN_CH'], FCFG['TCN_KERNEL'], FCFG['TCN_BLOCKS'], FCFG['DROPOUT'])
    TCN.load_state_dict(BFC['state_dict'])
    TCN.eval()

    def step_dz(xz, xc):
        """(WIN, L) z-window + (C,) case -> dz (L,). TCN inference, dropout OFF."""
        x = np.concatenate([xz, np.tile(xc, (len(xz), 1))], 1)      # (WIN, L+C)
        with torch.no_grad():
            dz = TCN(torch.from_numpy(x[None]).float())[0].numpy()
        return dz.astype(np.float32)

    def rollout(xz_start, xc, hmax):
        """autoregressive rollout — the model's own predictions are fed back.
        Returns (preds (hmax, L) z-space, n_clamped entries)."""
        z = np.array(xz_start, np.float32).copy()
        preds = np.empty((hmax, len(LIVE_FC)), np.float32)
        nclamp = 0
        for h in range(hmax):
            dz = step_dz(z, xc)
            zn = z[-1] + dz
            if FCFG.get('CLAMP_ROLLOUT', True):
                zc_ = np.clip(zn, Z_LO, Z_HI)
                nclamp += int((zc_ != zn).sum())
                zn = zc_
            z[:-1] = z[1:]
            z[-1] = zn
            preds[h] = zn
        return preds, nclamp

    if set(LIVE_FC) != set(LIVE_COLS):
        print(f"[fc] NOTE: forecaster live columns differ from the decoder's "
              f"({len(LIVE_FC)} vs {len(LIVE_COLS)}) - forecast rows are still valid "
              f"forecasts, only the renderable subset overlaps")
    print(f"[fc] TCN rebuilt: {sum(p.numel() for p in TCN.parameters()) / 1e3:.0f} k params, "
          f"WIN={WIN}, H_MAX={H_MAX_FC}, clamp range z in "
          f"[{float(Z_LO.min()):.1f}, {float(Z_HI.max()):.1f}], "
          f"onset threshold nbub > {ONSET_TH:g}")
    print(f"[fc] loaded from {os.path.basename(ART['fc']['path'])}")
else:
    print('[fc] no FC_TCN bundle -> boiling_forecast tool disabled (render-only)')

# ---- theta time series per file (streamed, cached, LRU 3) --------------------
_THETA_CACHE = {}               # stem -> (tab, full_flag)
_THETA_ORDER = []

def theta_table(stem, need=None):
    """theta rows (LIVE_FC + wallTemp + pos) for one simulation.
    need=None -> the WHOLE file (required for onset search); need=k -> the last
    k rows suffice (fast path for origin='end'). A cached full table serves any
    need by slicing; a cached tail table serves smaller needs."""
    if stem not in STEM2PATH:
        return None
    ent = _THETA_CACHE.get(stem)
    if ent is not None:
        tab, full = ent
        if full:
            return tab if need is None else tab.tail(need)
        if need is not None and len(tab) >= need:
            return tab.tail(need)
    path = STEM2PATH[stem]
    f, kd, kt = h5_open(path)
    n = int(f[kd].shape[0])
    _nf, wt, dt = file_meta(path)
    thot = t_hot_of_walltemp(wt)
    lo = 0 if need is None else max(0, n - max(int(need), WIN + 8))
    if n - lo > 400:
        print(f"[fc] streaming theta from {stem} ({n - lo} frames, first time "
              f"only - cached afterwards)")
    rows = []
    for pos in range(lo, n):                                # streamed frame by frame
        d = np.asarray(f[kd][pos], np.float32)
        T = np.asarray(f[kt][pos], np.float32)
        th = extract_theta(centre_crop(d, H, W), centre_crop(T, H, W), thot)
        rows.append({c: th[c] for c in LIVE_FC})
        rows[-1]['wallTemp'] = wt
        rows[-1]['pos'] = pos
        if (pos - lo) % 500 == 499:
            print(f"[fc]   {pos + 1}/{n}")
    tab = pd.DataFrame(rows)
    if not len(tab):
        return None
    _THETA_CACHE[stem] = (tab, need is None)
    _THETA_ORDER.append(stem)
    while len(_THETA_ORDER) > 3:                            # LRU: 3 files resident
        _THETA_CACHE.pop(_THETA_ORDER.pop(0), None)
    return tab

def _z_of_table(tab):
    """(n, L) z-space over LIVE_FC (forecaster normalization)."""
    return ((tab[LIVE_FC].values - np.array([TH_MU_FC[c] for c in LIVE_FC]))
            / np.array([TH_SD_FC[c] for c in LIVE_FC])).astype(np.float32)

def _case_z(wt):
    return np.array([(wt - TH_MU_FC[c]) / TH_SD_FC[c] for c in CASE_FC], np.float32)

def _phys_preds(preds_z):
    return {c: (preds_z[:, j] * TH_SD_FC[c] + TH_MU_FC[c]).astype(np.float32)
            for j, c in enumerate(LIVE_FC)}

def forecast(stem, origin='end', horizon=20):
    """TCN rollout from a real window of one simulation.
    origin: 'end' = the most recent WIN rows; 'onset' = the window just before
    the first bubble appears. Returns a dict."""
    # origin='end' only needs the last WIN rows -> tail fast path; 'onset'
    # searches for the first bubble, which needs the whole file.
    tab = theta_table(stem) if origin == 'onset' else theta_table(stem, need=WIN + 2)
    if tab is None or len(tab) < WIN + 1:
        return dict(ok=False, reason=f'no usable theta table for {stem!r}')
    horizon = int(min(max(1, horizon), H_MAX_FC))
    origin_effective = origin
    nb = tab['nbub'].values if 'nbub' in tab else None
    j0 = int(len(tab) - 1)
    if origin == 'onset' and nb is not None:
        on = np.flatnonzero(nb > 0)
        if len(on) and on[0] >= WIN:
            j0 = int(on[0])                               # window ends BEFORE the onset
        else:
            origin_effective = 'end'                       # no onset found - say so
    win = tab.iloc[j0 - WIN + 1: j0 + 1]
    xz = _z_of_table(win)
    xc = _case_z(float(tab['wallTemp'].iat[0]))
    preds_z, nclamp = rollout(xz, xc, horizon)
    phys = _phys_preds(preds_z)
    # onset + burst statistics on the predicted bubble count
    onset_step, episodes = None, []
    if 'nbub' in phys:
        hot = phys['nbub'] > ONSET_TH
        if hot.any():
            onset_step = int(np.flatnonzero(hot)[0])
            d = np.flatnonzero(np.diff(np.concatenate([[0], hot.view(np.int8), [0]])))
            episodes = [(int(a), int(b)) for a, b in zip(d[0::2], d[1::2])]
    period = None
    if len(episodes) >= 2:
        starts = [a for a, _ in episodes]
        period = float(np.mean(np.diff(starts)))
    return dict(ok=True, stem=stem, origin=origin, origin_effective=origin_effective,
                origin_pos=int(win['pos'].iat[-1]), horizon=horizon,
                nclamp=int(nclamp), onset_step=onset_step, episodes=episodes,
                burst_period=period, steps=np.arange(1, horizon + 1).tolist(),
                **{f'{c}': [float(v) for v in phys[c]] for c in phys},
                _phys=phys, _xz_end=xz)

if FC_AVAILABLE:
    _f = forecast(ROW_FILE[0], origin='end', horizon=16)
    _col0 = LIVE_FC[0]                    # no hardcoded column names in tests
    assert _f.get('ok') and len(_f[_col0]) == 16, \
        f'TCN rollout smoke failed: {_f.get("reason") or "bad trajectory length"}'
    assert all(np.isfinite(v).all() for v in _f['_phys'].values()), \
        'TCN rollout produced non-finite values'
    print(f"[fc] rollout smoke on {ROW_FILE[0]!r}: 16 steps over {len(LIVE_FC)} "
          f"columns, {_f['nclamp']} clamped entries, "
          f"onset_step={_f['onset_step']} - finite PASS")
print('[ok] CELL 4 done' + (' — TCN forecaster live' if FC_AVAILABLE else ''))


[fc] TCN rebuilt: 37 k params, WIN=32, H_MAX=80, clamp range z in [-3.7, 12.3], onset threshold nbub > 0.1
[fc] loaded from FC_F2_TCN_bundle_f15files_20261106_0927.pt
[fc] rollout smoke on 'Twall_107': 16 steps over 13 columns, 83 clamped entries, onset_step=0 - finite PASS
[ok] CELL 4 done — TCN forecaster live


## 5 · The option space, the model card, the merge, and question routing

What the LLM may ask for is derived from the bundles: the two supported
wall-temperature islands, the live descriptor columns with units and train-split
σ. Unspecified columns are auto-filled from the nearest retriever row; |z| is
clamped at 2.5 with per-column reporting; off-manifold wall temperatures are
flagged, never silently interpolated. The **model card** carries the validated
facts of the executed run (floors, per-column errors, dead knobs, chaos limit) —
it feeds the system prompt and every tool response, so honesty is enforced by the
plumbing. The merge labels each column *agree / partial / disputed* by the M4–M2
gap in σ; the merged range **is** the answer.

**Routing (v2):** the `INTENT_MAP` table maps every question type to its tool,
the calculation that tool runs, and the facts the answer must state —
render, forecast, conditions, sensitivity (two renders), nearest-real,
reliability (direct card answer, no tool) and refuse (not-available list). The
real LLM sees this table in its system prompt; the offline placeholder routes
with the same table; the self-check verifies the canonical questions land on
the right rows.

In [5]:
# CELL 5 · the option space, the model card, the two-model merge, and the
#          question-routing table
#
# The option space (what the LLM may ask for) is derived from the bundles, not
# hand-written. The model card carries the validated facts of the training run —
# it feeds the system prompt, the /card endpoint and every tool response, so the
# LLM's honesty is enforced by the plumbing rather than by the prompt alone.
# The INTENT_MAP below is the explicit routing layer: every question TYPE maps
# to the tool that answers it, the calculation that tool runs, and the facts the
# answer must state. The real LLM sees this table in its system prompt; the
# offline placeholder LLM routes with the same table, so every route is testable
# without a key.

# ---- supported conditions (from the M2 bundle: files + wall temperatures) -----
# The bundle's twall metadata can be stringly ('Twall_10'): parse robustly and
# fall back to the stem ('f8106@10C' -> 10). Every consumer works with floats.
COND_INFO = []
_unparsed = []
for stem, wt in zip(B2['files'], B2['twall']):
    wt = _wt_num(wt)
    if not np.isfinite(wt):
        wt = _wt_stem(stem)
    if not np.isfinite(wt):
        _unparsed.append(str(stem))
        fam = 'unknown'
    else:
        fam = 'boiling' if wt >= SVC_CFG['BOILING_TWALL_MIN'] else 'single_bubble'
    n = None
    if stem in STEM2PATH:
        n, _wt, _dt = file_meta(STEM2PATH[stem])
        if not np.isfinite(wt):                     # last resort: the file itself
            wt = _wt_num(_wt)
    COND_INFO.append(dict(stem=stem, wallTemp=float(wt), family=fam, n_frames=n,
                          in_index=stem in set(ROW_FILE)))
if _unparsed:
    print(f'[schema] WARNING: no wall temperature for {_unparsed} - excluded from '
          f'the islands; asks near them still work via the retriever')
_wts_sb = sorted(ci['wallTemp'] for ci in COND_INFO
                 if ci['family'] == 'single_bubble' and np.isfinite(ci['wallTemp']))
_wts_bo = sorted(ci['wallTemp'] for ci in COND_INFO
                 if ci['family'] == 'boiling' and np.isfinite(ci['wallTemp']))
ISLANDS = dict(single_bubble=(_wts_sb[0], _wts_sb[-1]) if _wts_sb else None,
               boiling=(_wts_bo[0], _wts_bo[-1]) if _wts_bo else None)
GAP = (_wts_sb[-1], _wts_bo[0]) if (_wts_sb and _wts_bo) else None

# units for every column the LLM can ask for: the known descriptor names plus
# every cond column of THIS bundle (the wall column may be named 'Twall' etc.)
UNITS = {**{f'alpha_{r}': 'fraction 0-1' for r in range(1, 6)},
         **{f'Tmean_{r}': 'degC' for r in range(1, 6)},
         **{f'f_hot_{r}': 'fraction 0-1' for r in range(1, 6)},
         'D32': 'px', 'nbub': 'count', 'wallTemp': 'degC'}
for _c in COND_COLS:
    UNITS.setdefault(_c, 'case value')
UNITS[WALL_COL] = 'degC'
RANGES = {c: (0.0, 1.0) for c in UNITS if 'fraction' in UNITS[c]}
ZMAX_ASK = 2.5                 # |z| clamp for asks, counted and reported
_WALL_KEYS = {'walltemp', 'wall_temp', 'twall', 'wall temperature'}  # ask aliases

def island_status(wt):
    """where a wall temperature sits relative to the supported islands."""
    if wt is None or not np.isfinite(float(wt)):
        return 'unspecified'
    wt = float(wt)
    for name, (lo, hi) in ISLANDS.items():
        if name and lo - 1e-9 <= wt <= hi + 1e-9:
            return 'inside ' + name
    if GAP and GAP[0] < wt < GAP[1]:
        return f'off-manifold (between the islands, {GAP[0]:g}-{GAP[1]:g} degC)'
    _iv = [x for x in ISLANDS.values() if x]
    lo = min(x[0] for x in _iv)
    hi = max(x[1] for x in _iv)
    return f'extrapolating beyond the data ({lo:g}-{hi:g} degC)'

def resolve_ask(partial):
    """partial {col: physical value} -> full ask (z + physical), flags, notes.
    Unspecified columns are auto-filled from the nearest retriever row; |z| is
    clamped at ZMAX_ASK; off-manifold wall temperatures are flagged, never
    silently interpolated."""
    notes = []
    clean = {}
    for k, v in (partial or {}).items():
        if k != WALL_COL and k.lower() in _WALL_KEYS:
            k = WALL_COL         # accept the generic name for the bundle's column
        if k not in UNITS:
            notes.append(f'{k}: unknown column, ignored (schema: {", ".join(COND_COLS)})')
            continue
        try:
            v = float(v)
        except (TypeError, ValueError):
            notes.append(f"{k}: not a number, ignored")
            continue
        if not np.isfinite(v):
            notes.append(f'{k}: not finite, ignored')
            continue
        if k in RANGES:
            if v < RANGES[k][0] or v > RANGES[k][1]:
                v = min(max(v, *RANGES[k]), RANGES[k][1])
                notes.append(f'{k}: clamped to {UNITS[k]}')
        if k == 'nbub':
            v = max(0.0, round(v))
        clean[k] = v
    # nearest retriever row for the auto-fill (by the specified columns)
    z_part = np.full(len(COND_COLS), np.nan, np.float32)
    for k, v in clean.items():
        z_part[COND_COLS.index(k)] = (v - TH_MU[k]) / TH_SD[k]
    spec = ~np.isnan(z_part)
    if spec.any():
        d = np.linalg.norm((M2COND[:, spec] - z_part[spec]) / max(np.sqrt(spec.sum()), 1),
                           axis=1)
        k0 = int(np.argmin(d))
    else:
        k0 = len(M2COND) // 2                     # deterministic middle anchor
    filled = [c for c in COND_COLS if c not in clean]
    z = M2COND[k0].copy()
    for c, v in clean.items():
        z[COND_COLS.index(c)] = (v - TH_MU[c]) / TH_SD[c]
    clamped = []
    for j, c in enumerate(COND_COLS):
        if abs(z[j]) > ZMAX_ASK:
            z[j] = np.clip(z[j], -ZMAX_ASK, ZMAX_ASK)
            clamped.append(c)
    phys = phys_of_z(z)
    flags = []
    wt = clean.get(WALL_COL, phys.get(WALL_COL, np.nan))
    st = island_status(float(wt))
    if 'off-manifold' in st:
        flags.append('off_manifold')
    if 'extrapolating' in st:
        flags.append('extrapolating')
    _, nn, _, _ = retrieve(z)
    if nn > 1.0:
        flags.append('far_from_support')
    return dict(ask_phys=phys, ask_z=z, auto_filled=filled, clamped=clamped,
                flags=flags, notes=notes + [f'wall temperature: {st}'],
                wall_status=st, nn=nn, anchor_row=k0)

# ---- agreement labels for the two-model merge --------------------------------
def agreement(gap_sigma):
    if gap_sigma < 0.5:
        return 'agree'
    if gap_sigma < 1.5:
        return 'partial'
    return 'disputed'

def merge_table(ask_phys, m4_theta, m2_phys):
    """per live column: ask | M4 | M2 (real) | merged range | gap sigma | verdict.
    M4 theta_hat arrives in FRAME_COLS order; m2_phys is a {col: value} dict."""
    rows = []
    for c in LIVE_COLS:
        a = float(ask_phys.get(c, np.nan))
        m4 = float(m4_theta[FRAME_COLS.index(c)])
        m2 = float(m2_phys.get(c, np.nan))
        lo, hi = (m4, m2) if m4 <= m2 else (m2, m4)
        gap = abs(m4 - m2) / max(TH_SD[c], 1e-9)
        rows.append(dict(column=c, unit=UNITS.get(c, ''), ask=a, m4=m4, m2=m2,
                         range_lo=float(lo), range_hi=float(hi),
                         gap_sigma=float(gap), verdict=agreement(gap)))
    counts = {v: sum(1 for r in rows if r['verdict'] == v)
              for v in ('agree', 'partial', 'disputed')}
    return rows, counts

# ---- the model card (validated facts of the executed 15-file run) -------------
if FIXTURE:
    _F = 'FIXTURE MODE — synthetic corpus, numbers are placeholders for format only.'
else:
    _F = None
MODEL_CARD = dict(
    fixture_note=_F,
    framing=('Two-stage surrogate for pool boiling: a 13-column physical descriptor '
             '(5 wall-normal regions x {void fraction, mean temperature, hot fraction}, '
             'Sauter diameter D32, bubble count) + wall temperature. Short-term '
             'forecasting (<= 20 steps of 0.1 s) is dynamic and skillful; long-range '
             'point forecasts of the void variables are chaos-limited and only their '
             'statistics (onset timing, burst period, plateaus) are trustworthy.'),
    floors=dict(real_next_test_pct=3.9, chaos_floor_h80_pct=45.5),
    m4=dict(test_rt_pct=11.7, test_rt_cal_pct=9.9, median=10.9, p90=22.4, blowups=8,
            edge_energy=0.96, per_col_pct=dict(Tmean='0.3-0.7', f_hot_1=0.7,
                                               alpha_2_5='5.6-12.7', alpha_1=30.6,
                                               nbub=34.4, D32=51.5),
            knobs=dict(alpha='gains 1-2.7, cross-leak ~1 (steerable, overshooting)',
                       Tmean='0.26-0.45', f_hot_1=0.25, nbub=1.21,
                       D32='-0.45 (wrong sign)', wallTemp='0.01 (dead knob)'),
            nngap_spearman=0.69),
    m2=dict(test_rt_pct=7.9, median=1.5, p90=25.2, blowups=1885,
            note='returns only frames that exist; error = distance to the support'),
    tcn=dict(params_k=37, receptive_field_steps=125, val_h20_pct=18.6,
             persistence_val_h20_pct=24.0, test_h20_pct=34.8,
             persistence_test_h20_pct=40.2, test_h80_pct=61.8,
             onset_mae_steps=14.6, burst_period_steps='26-30', early_bias='63-69%',
             event_f1=0.0, thermal_trend_acc=0.184),
    horizon_policy=('H <= 20 steps (2 s): point trajectories. H > 20: statistics only '
                    '(onset time, burst period, plateau ranges).'),
    not_available=['heat flux / heat transfer coefficient', 'pressure', 'refrigerant '
                   'properties', 'COP or system-level heat pump quantities', '3-D '
                   'geometry, wall material/roughness', 'anything between the two '
                   'wall-temperature islands (off-manifold)'],
)

def caveats_for(kind):
    """the model-card facts that must travel inside every tool response."""
    if FIXTURE:
        return ['FIXTURE MODE: synthetic data — every number is a placeholder.']
    if kind == 'render':
        return ['the wall-temperature knob is dead (control gain 0.01) — steer via '
                'the descriptor columns, not via the wall temperature',
                'D32 (51.5% test error, gain -0.45) and nbub (34.4%) are the least '
                'reliable columns; Tmean (0.3%) and f_hot_1 (0.7%) are the most reliable',
                f'{"-".join(f"{g:g}" for g in GAP)} degC lies between the two supported '
                'islands — off-manifold, flagged, never interpolated',
                'M2 returns only frames that actually exist in the simulations; the '
                'merged range is the honest uncertainty, not an average']
    return ['beyond 20 steps only statistics are trustworthy (chaos floor 45.5% at '
            'H=80 — the divergence of the nearest real trajectory)',
            'onset timing error is ~14.6 steps against a burst period of 26-30 steps; '
            '63-69% of predicted onsets fire early',
            'nucleation event detection is unreliable (event F1 = 0.0) — treat '
            'individual predicted bubbles beyond the short horizon as indicative',
            'thermal columns stay predictable (0.3-1% at H=80); void columns do not']

# ---- the question-routing table (shared by the system prompt, the offline
#      placeholder LLM and the self-check; order = precedence) ------------------
INTENT_MAP = [
    dict(intent='not_available',
         patterns=('heat flux', 'heat transfer coefficient', 'cop', 'pressure',
                   'refrigerant', 'efficiency', 'geometry', 'roughness', 'material'),
         tool=None,
         process='refuse with the not-available list; offer the closest '
                 'tool-based alternative (descriptor or forecast questions)',
         must_state=('what is not available', 'the closest available alternative')),
    dict(intent='reliability',
         patterns=('reliab', 'error', 'accura', 'trust', 'uncertain', 'floor',
                   'limitation', 'how good', 'confidence', 'valid', 'fail'),
         tool=None,
         process='answer directly from the model card - no tool call needed '
                 '(floors, per-column errors, dead knobs, chaos limit)',
         must_state=('the floors (real-next 3.9%, chaos 45.5% at H=80)',
                     'per-column reliability classes',
                     'the dead wall-temperature knob', 'event F1 = 0')),
    dict(intent='forecast',
         patterns=('next', 'forecast', 'predict', 'future', 'evolve', 'happen',
                   'when', 'burst', 'period', 'onset', 'plateau', 'long-run',
                   'look ahead', 'how often'),
         tool='boiling_forecast',
         process='pick the simulation (nearest wall temperature, or a named stem) '
                 '-> choose origin (end = latest state, onset = just before the '
                 'first bubble) -> TCN rollout feeding back its own predictions '
                 '-> point trajectory to 20 steps, statistics beyond',
         must_state=('the simulation and origin used',
                     'point values only up to 20 steps',
                     'onset step (its MAE is 14.6 steps) and burst period',
                     'the chaos-floor caveat for anything beyond 20 steps')),
    dict(intent='conditions',
         patterns=('which cases', 'list', 'conditions', 'corpus', 'what data',
                   'how many', 'files', 'simulations', 'temperature range',
                   'what is in the data'),
         tool='list_conditions',
         process='scan the discovered corpus: family, wall temperature, frame '
                 'count, retriever coverage -> conditions table + islands',
         must_state=('the two islands', 'the number of simulations',
                     'the gap between the islands is off-manifold')),
    dict(intent='sensitivity',
         patterns=('what if', 'what happens if', 'increase', 'decrease', 'double',
                   'halve', 'change', 'sensitivity', 'effect of', 'more bubbles',
                   'fewer bubbles', 'larger', 'smaller', 'vary'),
         tool='boiling_render twice (baseline + modified ask)',
         process='call boiling_render on the baseline ask and on the modified '
                 'ask (two calls) -> compare the two merged tables -> report '
                 'which columns moved and which did not',
         must_state=('the two asks', 'the columns that moved (and by how much)',
                     'the columns that did NOT move (the knobs are dead for them)',
                     'alpha knobs cross-leak ~1; D32 gain is -0.45 (wrong sign)')),
    dict(intent='nearest_real',
         patterns=('nearest real', 'is there data', 'actual case', 'real case',
                   'closest simulation', 'real example', 'ground truth'),
         tool='boiling_render (the M2 half is the answer)',
         process='resolve the ask -> M2 retrieves the nearest REAL frame -> '
                 'report its file, time and descriptor distance; M4 is shown '
                 'alongside for the comparison',
         must_state=('the retrieved file and frame time',
                     'the distance in sigma (how well the corpus covers the ask)',
                     'the merged range as usual')),
    dict(intent='render',
         patterns=(),               # default: any other state question
         tool='boiling_render',
         process='resolve the ask (auto-fill unspecified columns, clamp, flag '
                 'off-manifold) -> M4 renders a frame AND M2 retrieves the '
                 'nearest real frame -> both re-analysed -> merged per-column '
                 'range with verdicts',
         must_state=('the merged range per column (never a single model)',
                     'round-trip % and NNgap in sigma',
                     'verdict counts (agree/partial/disputed)',
                     'flags, caveats and the floor line')),
]

def route_intent(message):
    """deterministic pattern router over INTENT_MAP (precedence order).
    Used by the offline placeholder LLM and by the self-check; the real LLM
    receives the same table as text and is expected to route equivalently."""
    low = (message or '').lower()
    for row in INTENT_MAP:
        if any(p in low for p in row['patterns']):
            return row
    return INTENT_MAP[-1]           # render is the default intent

print(f"[schema] wall column {WALL_COL!r} (ask aliases accepted); "
      f"twall entries parsed: "
      + ", ".join(f"{ci['stem']}={ci['wallTemp']:g}" for ci in COND_INFO
                  if np.isfinite(ci['wallTemp'])))
print(f"[schema] supported islands: "
      + "; ".join(f"{k} {v[0]:g}-{v[1]:g} degC" for k, v in ISLANDS.items() if v))
print(f"[schema] {len(LIVE_COLS)} live descriptor columns + {WALL_COL}; "
      f"asks clamped at |z| <= {ZMAX_ASK} with per-column reporting")
print(f"[route] {len(INTENT_MAP)} question intents -> "
      f"{sorted({r['tool'].split(' ')[0] for r in INTENT_MAP if r['tool']})} "
      f"(+ direct model-card answers)")
print('[ok] CELL 5 done — option space, model card, merge and routing defined')


[schema] wall column 'wallTemp' (ask aliases accepted); twall entries parsed: Twall_10=10, Twall_100=100, Twall_107=107, Twall_13=13, Twall_14=14, Twall_15=15, Twall_17=17, Twall_18=18, Twall_19=19, Twall_21=21, Twall_22=22, Twall_23=23, Twall_26=26, Twall_75=75, Twall_98=98
[schema] supported islands: single_bubble 10-26 degC; boiling 75-107 degC
[schema] 13 live descriptor columns + wallTemp; asks clamped at |z| <= 2.5 with per-column reporting
[route] 7 question intents -> ['boiling_forecast', 'boiling_render', 'list_conditions'] (+ direct model-card answers)
[ok] CELL 5 done — option space, model card, merge and routing defined


## 6 · The tools

One composite tool per question type. `boiling_render` runs **both** models in one
atomic call — M4 renders a frame, M2 retrieves the nearest real frame, both are
re-analysed and merged per column — so the LLM cannot answer with a single model or
invent its own merge. The two heat-map images are saved on **one shared color
scale** (vmin/vmax span both frames; the range is printed on each colorbar and
travels in the JSON so the page can caption it) — the same shade means the same
temperature in the M4 and the M2 panel, making the frames directly comparable.
`boiling_forecast` rolls the TCN (point trajectory to 20 steps, statistics beyond,
onset and burst stats, chaos-floor caveat inside) and reports the effective origin
when a pre-onset window does not exist. `list_conditions` reports the corpus.
`condense()` turns any tool result into the short deterministic trace lines the
browser shows as "thinking steps" — never LLM text, so the trace cannot
hallucinate; it is defensive by design (a malformed result degrades one line,
never the call). `reply_summary()` builds the closing **"In short"** lines the
same way: plain sentences that explain what the results mean (which models ran,
how far apart they are, the largest gap, onset and burst numbers, the doubled-ask
comparison), derived only from the tool JSONs — so the summary cannot hallucinate
either and appears in both online and offline answers.

In [6]:
# CELL 6 · the tools — boiling_render (M4 AND M2, merged), list_conditions,
#          boiling_forecast (TCN) — plus the trace-line condenser
#
# One composite tool per question type: boiling_render runs BOTH models in one
# atomic call and merges them as a range, so the LLM cannot answer with a single
# model or invent its own merge. Every response carries its caveats inside.
import os, uuid
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
for _fp in ('/usr/share/fonts/truetype/chinese/NotoSansSC-Regular.ttf',
            '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'):
    if os.path.exists(_fp):
        fm.fontManager.addfont(_fp)
plt.rcParams['font.sans-serif'] = ['Noto Sans SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

IMG_DIR = os.path.join(RUNTIME_DIR, 'static', 'img')
os.makedirs(IMG_DIR, exist_ok=True)

def _png(fig):
    """save a figure under the static dir; return its web path."""
    name = f'{uuid.uuid4().hex[:12]}.png'
    fig.savefig(os.path.join(IMG_DIR, name), dpi=110, facecolor='white')
    plt.close(fig)
    return f'/static/img/{name}'

def _T_image(T, title, vmin, vmax):
    """one temperature frame; vmin/vmax cover BOTH frames (shared scale: the
    same shade means the same temperature in the M4 and the M2 panel, so the
    two frames are directly comparable). Extra layout padding keeps the
    title/colorbar fully inside the canvas - a title that touches the figure
    edge is clipped in the saved PNG, which looks exactly like a cropped image
    in the browser."""
    fig, ax = plt.subplots(figsize=(3.4, 4.6), constrained_layout=True)
    fig.set_constrained_layout_pads(w_pad=0.1, h_pad=0.15)
    im = ax.imshow(T, cmap='inferno', vmin=vmin, vmax=vmax, aspect='equal')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title, fontsize=9, pad=6)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03,
                 label=f'T degC (shared {vmin:.1f}-{vmax:.1f})')
    return _png(fig)

def _seconds(steps, dt=0.1):
    return steps * dt

def tool_boiling_render(ask_partial):
    """M4 renders a frame for the ask AND M2 retrieves the nearest real frame;
    both are re-analysed and merged per column as a range."""
    res = resolve_ask(ask_partial)
    z = res['ask_z']
    thot = t_hot_of_cond_row(z)
    d, T, _ = generate(z[None])
    th_hat = np.array([[extract_theta(d[0], T[0], thot)[c] for c in FRAME_COLS]],
                      np.float32)                      # raw round-trip
    th_cal = apply_cal(th_hat)[0]                      # reporting affine
    m4_theta = {c: float(th_cal[FRAME_COLS.index(c)]) for c in FRAME_COLS}
    m4_raw = {c: float(th_hat[0][FRAME_COLS.index(c)]) for c in FRAME_COLS}
    k, dist, stem, pos = retrieve(z)
    m2_phys = phys_of_z(M2COND[k])
    _n, _wt, dt = file_meta(STEM2PATH[stem])
    rows, counts = merge_table(res['ask_phys'], th_cal, m2_phys)
    live_set = set(LIVE_COLS)
    _sd = np.array([TH_SD[c] for c in FRAME_COLS])
    _ask_f = np.array([res['ask_phys'].get(c, np.nan) for c in FRAME_COLS])
    _live = np.array([c in live_set for c in FRAME_COLS])
    rt_cal = 100 * float(np.nanmean(np.abs(th_cal[_live] - _ask_f[_live]) / _sd[_live]))
    rt_raw = 100 * float(np.nanmean(np.abs(th_hat[0][_live] - _ask_f[_live]) / _sd[_live]))
    d_real, T_real = frame_of(STEM2PATH[stem], pos)
    wt_ask = float(res['ask_phys'].get(WALL_COL, np.nan))
    _wt_lab = f'{wt_ask:.0f}' if np.isfinite(wt_ask) else 'n/a'
    # ONE shared color scale for both panels (details live in the HTML caption)
    vmin = round(float(min(T.min(), T_real.min())), 1)
    vmax = round(float(max(T.max(), T_real.max())), 1)
    img_m4 = _T_image(T[0], f'M4 - generated - Twall {_wt_lab} degC', vmin, vmax)
    img_m2 = _T_image(T_real, f'M2 - real - {stem}', vmin, vmax)
    nb_ratio = (m4_theta['nbub'] / res['ask_phys']['nbub']
                if res['ask_phys'].get('nbub') else None)
    return dict(
        tool='boiling_render',
        ask={c: round(float(v), 4) for c, v in res['ask_phys'].items()},
        auto_filled=res['auto_filled'], clamped=res['clamped'], flags=res['flags'],
        wall_status=res['wall_status'], notes=res['notes'],
        nn_sigma=round(float(dist), 3),
        m4=dict(round_trip_pct=round(rt_cal, 1), round_trip_raw_pct=round(rt_raw, 1),
                theta={c: round(v, 3) for c, v in m4_theta.items() if c in live_set},
                theta_raw={c: round(v, 3) for c, v in m4_raw.items() if c in live_set}),
        m2=dict(stem=stem, frame_pos=int(pos), t_s=round(pos * dt, 1), row=int(k),
                distance_sigma=round(float(dist), 3),
                theta={c: round(float(m2_phys[c]), 3) for c in live_set}),
        merged=dict(rows=[{kk: (round(vv, 4) if isinstance(vv, float) else vv)
                           for kk, vv in r.items()} for r in rows],
                    counts=counts,
                    summary=f"{counts['agree']}/{len(rows)} columns agree (<0.5 sigma), "
                            f"{counts['disputed']} disputed (>=1.5 sigma)"),
        images=dict(m4=img_m4, m2=img_m2,
                    m4_range=[vmin, vmax], m2_range=[vmin, vmax],
                    shared_scale=True),
        nbub_ratio=(round(float(nb_ratio), 2) if nb_ratio is not None else None),
        floor_line=f'floor: real-next 3.9% (test) - this ask NNgap {dist:.2f} sigma',
        caveats=caveats_for('render'),
    )

def tool_list_conditions():
    rows = [dict(stem=ci['stem'], wallTemp=f"{ci['wallTemp']:g}",
                 family=ci['family'],
                 n_frames=ci['n_frames'] if ci['n_frames'] else 'n/a',
                 in_retriever_index='yes' if ci['in_index'] else 'no')
            for ci in COND_INFO]
    islands = '; '.join(f"{k} {v[0]:g}-{v[1]:g} degC" for k, v in ISLANDS.items() if v)
    return dict(tool='list_conditions', rows=rows,
                islands=islands,
                note='the retriever index covers the training frames of each file; '
                     'every file is a separate simulation (one wall temperature)',
                caveats=caveats_for('render'))

def tool_boiling_forecast(wallTemp=None, stem=None, horizon=20, origin='end'):
    """TCN rollout from a real window of the chosen simulation."""
    if not FC_AVAILABLE:
        return dict(tool='boiling_forecast', ok=False,
                    reason='no FC_TCN bundle discovered - forecast disabled')
    if stem is None:
        cands = [ci for ci in COND_INFO if ci['in_index']]
        if not cands:
            cands = COND_INFO
        if wallTemp is None or not np.isfinite(float(wallTemp)):
            ci = cands[len(cands) // 2]
        else:
            ci = min(cands, key=lambda c: abs(c['wallTemp'] - float(wallTemp)))
        stem = ci['stem']
    if stem not in STEM2PATH:
        return dict(tool='boiling_forecast', ok=False,
                    reason=f'unknown simulation {stem!r}')
    horizon = int(min(max(1, int(horizon)), H_MAX_FC))
    origin = origin if origin in ('end', 'onset') else 'end'
    _t0 = time.time()
    fr = forecast(stem, origin=origin, horizon=horizon)
    if not fr.get('ok'):
        return dict(tool='boiling_forecast', ok=False, reason=fr.get('reason'))
    _extract_s = time.time() - _t0
    _n, _wt, dt = file_meta(STEM2PATH[stem])
    phys = fr.pop('_phys')
    fr.pop('_xz_end', None)
    picks = [c for c in ('alpha_1', 'alpha_3', 'Tmean_1', 'nbub') if c in phys]
    picks += [c for c in LIVE_FC if c not in picks][:4 - len(picks)]
    fig, axes = plt.subplots(2, 2, figsize=(9.5, 6.2), constrained_layout=True)
    # head padding: without it the suptitle's ascenders touch the canvas edge and
    # arrive CLIPPED in the saved PNG (looks like a cropped chart in the browser)
    fig.set_constrained_layout_pads(w_pad=0.08, h_pad=0.25)
    for ax, c in zip(axes.ravel(), picks):
        ax.plot(_seconds(1, dt) + np.arange(horizon) * dt,
                np.asarray(phys[c]), lw=1.8, color='#3557a0')
        ax.set_title(f'{c} [{UNITS.get(c, "")}]', fontsize=10)
        ax.grid(alpha=0.3)
        if horizon > 20:
            ax.axvline(20 * dt, color='#b3541e', ls='--', lw=1.2)
            ax.axvspan(20 * dt, horizon * dt, color='#b3541e', alpha=0.06)
        if fr['onset_step'] is not None:
            ax.axvline((fr['onset_step'] + 1) * dt, color='#2e8b57', ls=':', lw=1.6)
    axes.ravel()[0].set_ylabel('descriptor value')
    for ax in axes[-1]:
        ax.set_xlabel('t (s)')
    axes.ravel()[0].set_title(
        axes.ravel()[0].get_title() +
        f'  -  TCN from {stem}, origin {fr["origin_effective"]}', fontsize=10)
    _l = (f'green dotted: predicted onset (step {fr["onset_step"] + 1})' if
          fr['onset_step'] is not None else 'no onset predicted within the horizon')
    fig.suptitle(f'TCN forecast, H = {horizon} steps ({horizon * dt:.1f} s) - '
                 f'dashed: H = 20 point/statistics boundary; {_l}', fontsize=9.5)
    img = _png(fig)
    _half_col = next((c for c in phys if c.startswith('alpha_')), None)
    half = phys[_half_col][horizon // 2:] if _half_col else None
    stats = dict(stem=stem, origin=origin,
                 origin_effective=fr['origin_effective'],
                 origin_pos=fr['origin_pos'],
                 horizon_steps=horizon, horizon_s=round(horizon * dt, 2),
                 dt_s=dt, clamped_entries=fr['nclamp'],
                 onset_step=(fr['onset_step'] + 1 if fr['onset_step'] is not None
                             else None),
                 onset_s=(round((fr['onset_step'] + 1) * dt, 2)
                          if fr['onset_step'] is not None else None),
                 burst_period_steps=(round(fr['burst_period'], 1)
                                     if fr['burst_period'] else None),
                 n_burst_episodes=len(fr['episodes']),
                 plateau=({f'{_half_col} (2nd half)': f'{float(np.mean(half)):.3f} '
                          f'± {float(np.std(half)):.3f}'} if _half_col else None))
    if origin == 'onset' and fr['origin_effective'] != 'onset':
        stats['note'] = ('no pre-onset window found (bubbles present from the first '
                         'frames) - rolled out from the latest state instead')
    if _extract_s > 5:
        stats['note'] = (stats.get('note', '') +
                         f' (theta extraction {_extract_s:.0f} s, first time for '
                         f'this file - cached afterwards)').strip()
    traj = {c: [round(float(v), 4) for v in np.asarray(phys[c])]
            for c in LIVE_FC if c in ('alpha_1', 'alpha_3', 'Tmean_1', 'nbub')}
    return dict(tool='boiling_forecast', ok=True, stats=stats, trajectory=traj,
                image=img, horizon_policy=MODEL_CARD['horizon_policy'],
                caveats=caveats_for('forecast'))

# ---- deterministic trace lines for the thinking-steps panel -------------------
def condense(result):
    """tool result -> short human lines for the step-by-step panel (never LLM text,
    so the trace cannot hallucinate). Defensive by design: a missing key degrades
    the line, never the call."""
    kind = result.get('tool')
    out = []
    try:
        if kind == 'boiling_render' and result.get('merged'):
            n_fill = len(result.get('auto_filled') or [])
            _wt = result.get('ask', {}).get(WALL_COL, float('nan'))
            out.append(f"ask resolved - {WALL_COL} "
                       f"{_wt if _wt == _wt else 0:.0f} degC, "
                       f"{n_fill} unspecified columns auto-filled from the nearest "
                       f"training row, {len(result.get('clamped') or [])} clamped")
            out.append(f"M4 generated a frame - round-trip "
                       f"{result.get('m4', {}).get('round_trip_pct', '?')}%, NNgap "
                       f"{result.get('nn_sigma', '?')} sigma")
            out.append(f"M2 retrieved the nearest real frame - "
                       f"{result.get('m2', {}).get('stem', '?')} "
                       f"at t = {result.get('m2', {}).get('t_s', '?')} s "
                       f"(distance {result.get('m2', {}).get('distance_sigma', '?')}"
                       f" sigma)")
            out.append(f"merged - {result['merged'].get('summary', '?')}")
        elif kind == 'list_conditions':
            out.append(f"scanned the corpus - {len(result.get('rows') or [])} "
                       f"simulations, islands: {result.get('islands', '?')}")
        elif kind == 'boiling_forecast':
            s = result.get('stats', {})
            out.append(f"TCN rollout from {s.get('stem')} (origin "
                       f"{s.get('origin_effective', s.get('origin'))}) - "
                       f"{s.get('horizon_steps')} steps, "
                       f"{s.get('clamped_entries')} clamped entries")
            out.append(f"onset predicted at step {s.get('onset_step')} "
                       f"({s.get('onset_s')} s); burst period "
                       f"{s.get('burst_period_steps')} steps" if s.get('onset_step')
                       else f"no onset predicted within {s.get('horizon_steps')} steps")
            out.append("horizon policy - point trajectory up to 20 steps, statistics "
                       "beyond (chaos floor 45.5% at H=80)")
        elif result.get('ok') is False:
            out.append(f"tool refused - {result.get('reason')}")
        else:
            out.append(f"{kind or 'tool'} returned a result (no condensed trace)")
    except Exception as e:
        out.append(f'tool trace unavailable ({e})')
    return out

# ---- deterministic closing lines: "what do these results mean" ------------------
def reply_summary(renders, forecasts, lists_):
    """the reply summary shown at the END of every chat answer: plain sentences
    that explain the results, built ONLY from the tool JSONs (never LLM text,
    so the summary cannot hallucinate - same philosophy as condense())."""
    out = []

    def _row(rows, col):
        return next((r for r in rows if r.get('column') == col), None)

    for r in renders:
        try:
            ask = r.get('ask', {})
            wt = ask.get(WALL_COL)
            counts = r.get('merged', {}).get('counts', {}) or {}
            n = len(r.get('merged', {}).get('rows') or [])
            m2, m4 = r.get('m2', {}), r.get('m4', {})
            wt_s = f'{wt:g}' if isinstance(wt, (int, float)) and wt == wt else '?'
            out.append(f"boiling_render at {WALL_COL} = {wt_s} degC: M4 generated a "
                       f"frame (round-trip {m4.get('round_trip_pct', '?')}% of the "
                       f"theta scale) and M2 retrieved the nearest real frame, "
                       f"{m2.get('stem', '?')} at t = {m2.get('t_s', '?')} s "
                       f"({r.get('nn_sigma', '?')} sigma from the ask).")
            out.append(f"the two models agree on {counts.get('agree', '?')}/{n} live "
                       f"columns ({counts.get('disputed', 0)} disputed) - the answer "
                       f"is the MERGED range per column, never one model's number; "
                       f"the two heat maps share one color scale (the same shade "
                       f"means the same temperature in both panels).")
            worst = max(r.get('merged', {}).get('rows') or [],
                        key=lambda row: row.get('gap_sigma') or 0, default=None)
            if worst is not None:
                out.append(f"largest model gap: {worst['column']} "
                           f"({worst.get('gap_sigma')} sigma, {worst.get('verdict')}) "
                           f"- merged {worst.get('range_lo')} to "
                           f"{worst.get('range_hi')} {worst.get('unit')}.")
        except Exception as e:
            out.append(f'render summary unavailable ({e})')

    for f in forecasts:
        try:
            if not f.get('ok'):
                out.append(f"boiling_forecast: {f.get('reason')}")
                continue
            s = f.get('stats', {}) or {}
            out.append(f"boiling_forecast from {s.get('stem')} (origin "
                       f"{s.get('origin_effective', s.get('origin'))}): "
                       f"{s.get('horizon_steps')} steps = {s.get('horizon_s')} s of "
                       f"TCN rollout from a real window.")
            if s.get('onset_step') is not None:
                _bp = s.get('burst_period_steps')
                out.append(f"first bubble (alpha onset) predicted at step "
                           f"{s.get('onset_step')} - t = {s.get('onset_s')} s; burst "
                           f"period "
                           + (f"{_bp} steps" if _bp is not None else 'not measurable')
                           + f" ({s.get('n_burst_episodes')} episodes).")
            else:
                out.append("no bubble onset within the horizon "
                           f"({s.get('horizon_steps')} steps).")
            if s.get('plateau'):
                _k, _v = next(iter(s['plateau'].items()))
                out.append(f"second-half plateau: {_k} = {_v} (mean +/- sd) - beyond "
                           "20 steps only such statistics are meaningful, not point "
                           "trajectories (chaos floor).")
        except Exception as e:
            out.append(f'forecast summary unavailable ({e})')

    for l in lists_:
        if l.get('tool') != 'list_conditions':
            continue
        try:
            out.append(f"the corpus holds {len(l.get('rows') or [])} simulations; "
                       f"wall-temperature islands: {l.get('islands', '?')}. "
                       "Every file is a separate simulation - the retriever index "
                       "covers their training frames.")
        except Exception as e:
            out.append(f'conditions summary unavailable ({e})')

    # sensitivity (two renders in one reply): what changed, and what responded
    if len(renders) == 2:
        try:
            a, b = renders
            ka, kb = a.get('ask', {}), b.get('ask', {})
            diff = {c: (ka[c], kb[c]) for c in ka if c in kb and ka[c] != kb[c]}
            if diff:
                parts = ', '.join(f"{c} {v0:g} -> {v1:g}"
                                  for c, (v0, v1) in diff.items()
                                  if isinstance(v0, (int, float))
                                  and isinstance(v1, (int, float)))
                out.append(f"comparison of the two renders ({parts}):")
                ra = a.get('merged', {}).get('rows') or []
                rb = b.get('merged', {}).get('rows') or []
                for c in diff:
                    r0, r1 = _row(ra, c), _row(rb, c)
                    if r0 and r1:
                        out.append(f"  {c} merged {r0['range_lo']}-{r0['range_hi']} -> "
                                   f"{r1['range_lo']}-{r1['range_hi']} "
                                   f"{r1['unit']}")
                for col in ('D32', 'nbub'):        # the interesting responders
                    r0, r1 = _row(ra, col), _row(rb, col)
                    if r0 and r1 and (r0['range_lo'] != r1['range_lo']
                                      or r0['range_hi'] != r1['range_hi']):
                        out.append(f"  response {col}: {r0['range_lo']}-"
                                   f"{r0['range_hi']} -> {r1['range_lo']}-"
                                   f"{r1['range_hi']} {r1['unit']}")
        except Exception as e:
            out.append(f'comparison summary unavailable ({e})')
    return out

TOOLS = dict(boiling_render=tool_boiling_render,
             list_conditions=tool_list_conditions,
             boiling_forecast=tool_boiling_forecast)

# ---- smoke test ---------------------------------------------------------------
_r = tool_boiling_render({'wallTemp': (ISLANDS['single_bubble'] or (0, 1))[0]
                          if ISLANDS['single_bubble'] else 20.0})
assert len(_r['merged']['rows']) == len(LIVE_COLS) and _r['images']['m4'] and \
    _r['images']['m2'], 'render tool smoke failed'
print(f"[tools] render smoke: RT {_r['m4']['round_trip_pct']}%, "
      f"NNgap {_r['nn_sigma']} sigma, {_r['merged']['summary']}")
if FC_AVAILABLE:
    _f = tool_boiling_forecast(horizon=16)
    assert _f['ok'] and _f['image'], 'forecast tool smoke failed'
    print(f"[tools] forecast smoke: {_f['stats']['stem']} "
          f"H={_f['stats']['horizon_steps']}, onset step "
          f"{_f['stats']['onset_step']}, clamps {_f['stats']['clamped_entries']}")
print('[ok] CELL 6 done — tools live: ' + ', '.join(TOOLS))


[tools] render smoke: RT 6.3%, NNgap 0.104 sigma, 13/13 columns agree (<0.5 sigma), 0 disputed (>=1.5 sigma)
[tools] forecast smoke: Twall_17 H=16, onset step 1, clamps 102
[ok] CELL 6 done — tools live: boiling_render, list_conditions, boiling_forecast


## 7 · The LLM client + the agentic loop

The client speaks plain `chat/completions` with function calling against
`LLM_BASE_URL` — same setup as the user's working standalone notebook: Bearer
auth, `{model, messages, temperature, max_tokens}` with `tools`/`tool_choice`
sent only when tools are present, the key checked **before** any request
(placeholder detection with a fix-it message), only 429/5xx/network retried,
any 4xx treated as permanent with the **server's own JSON message quoted
verbatim** (so a 400 says what the endpoint disliked instead of "Bad
Request"), and `llm_ping()` as the connectivity + authentication check run at
cell time. The **system prompt carries the routing table** from CELL 5, so per
question type the LLM follows the documented process and must-state list
instead of improvising; reliability questions are answered directly from the
card (no tool call). The agentic loop runs at most four rounds: plan → validate
args (tolerantly: `arguments` or `args`) → execute tools → append the
structured results (with caveats) → answer, forces a direct tool-free answer
if the round budget is spent, and trims the session history on user-message
boundaries so an assistant tool-call is never split from its tool result.
Without an API key the same loop runs on a scripted placeholder LLM that
routes with the same intent table and builds its answers from the real tool
results — that is the offline test mode. Every conversation is appended to
`boildesc_runtime/audit.jsonl`, and **every loop step is mirrored into the
PROGRESS registry** so the browser can show live thinking feedback.

In [7]:
# CELL 7 · the LLM client + the agentic loop
#
# Function calling via POST {LLM_BASE_URL}/chat/completions — the wire format
# ported as-is from the user's working standalone notebook (boildesc_llm_service5):
# Authorization: Bearer <key>, body {model, messages, temperature, max_tokens,
# tools, tool_choice} with tools sent only when present. The client is
# provider-agnostic (Google AI Studio / Z.ai / BigModel / local vLLM differ only
# in the three LLM_* settings), checks the key BEFORE the first request, retries
# only transient trouble (429/5xx/network), treats 4xx as permanent and always
# quotes the server's own JSON message, so a 400 says what the endpoint disliked
# instead of "Bad Request". llm_ping() is the connectivity+auth check to run
# before chatting.
# Without an API key the loop runs on a scripted placeholder LLM that routes with
# the SAME intent table as the system prompt (offline testing, fixture smoke).
# Every conversation is appended to a JSONL audit log, and every loop step is
# published to the PROGRESS registry so the browser can show live "thinking"
# feedback while /chat is in flight.
#
# v2: the system prompt carries the explicit ROUTING table (question type ->
# tool -> calculation -> facts the answer must state, from INTENT_MAP), so the
# LLM's routing follows the documented plan instead of improvising. The
# placeholder LLM routes every intent the same way, and reliability questions
# are answered directly from the model card (no tool call) in both modes.
import requests

# ---- key diagnostics (never prints the key itself) ----------------------------
_PLACEHOLDER_KEYS = ("", "sk-...", "sk-replace-me", "ky...", "your-api-key",
                     "paste-your-key-here", "replace-me", "test")

def _key_state():
    """describe LLM_API_KEY for diagnostics — length and prefix only, never the key"""
    k = (LLM_API_KEY or "").strip()
    if not k:
        return "EMPTY (no key was ever pasted)"
    if k.lower() in _PLACEHOLDER_KEYS or "..." in k or "replace" in k.lower():
        return f"a placeholder ({k[:6]!r}... is not a real key)"
    return f"set, {len(k)} characters, starts with {k[:4]!r}"

def _key_is_real():
    k = (LLM_API_KEY or "").strip()
    return (bool(k) and "..." not in k and "replace" not in k.lower()
            and k.lower() not in _PLACEHOLDER_KEYS)

def llm_chat(messages, tools=None, temperature=0.2, max_tokens=1500):
    """one chat-completions call; returns the assistant message dict.
    Same setup as the user's working standalone notebook: any OpenAI-compatible
    endpoint, Bearer auth, tools only when present, 4xx permanent with the
    server's message quoted verbatim, only 429/5xx/network retried."""
    if not _key_is_real():
        raise RuntimeError(
            "LLM_API_KEY is " + _key_state() + " — no request was sent.\n"
            "Fix: paste the key into .env next to this notebook (LLM_API_KEY=...),\n"
            "then restart the kernel and run all cells. The key must belong to the\n"
            "platform named in LLM_BASE_URL (Google AI Studio / Z.ai / BigModel keys\n"
            "are not interchangeable).")
    key = (LLM_API_KEY or "").strip()
    url = f"{LLM_BASE_URL.rstrip('/')}/chat/completions"
    body = {"model": LLM_MODEL, "messages": messages,
            "temperature": temperature, "max_tokens": max_tokens}
    if tools:
        body["tools"] = tools
        body["tool_choice"] = "auto"

    def _server_said(r):
        """the provider's own message (JSON body preferred) for readable errors"""
        try:
            return json.dumps(r.json(), ensure_ascii=False)[:400]
        except Exception:
            return (r.text or "")[:400]

    last_err = None
    for attempt in range(3):
        try:
            r = requests.post(url, headers={"Authorization": f"Bearer {key}"},
                              json=body, timeout=180)
            if r.status_code in (429, 500, 502, 503):     # transient -> retry
                last_err = f"HTTP {r.status_code}: {_server_said(r)}"
                time.sleep(2 ** attempt)
                continue
            if r.status_code >= 400:                      # permanent -> stop now
                hint = ""
                if r.status_code in (401, 403):
                    hint = ("\nWhat to check, in order:\n"
                            "  1. the key itself — " + _key_state() + ";\n"
                            "     re-copy it from the console: no spaces, no quotes\n"
                            "  2. platform match — the key must belong to the platform\n"
                            "     named in LLM_BASE_URL (keys are not interchangeable)\n"
                            "  3. the account's credit/quota (billing page)\n"
                            "  4. the model id — LLM_MODEL must exist on this endpoint")
                elif r.status_code == 404:
                    hint = ("\nLLM_BASE_URL looks wrong — it must stop before "
                            "'/chat/completions' (e.g. .../v1beta/openai).")
                elif r.status_code == 400:
                    hint = ("\nA 400 usually means LLM_MODEL is not a model id this "
                            "endpoint serves, or the endpoint rejected the request "
                            "schema — the server's message above says which.")
                raise RuntimeError(
                    f"LLM endpoint answered HTTP {r.status_code} at {url}\n"
                    f"server said: {_server_said(r)}{hint}")
            data = r.json()
            if "choices" not in data:
                raise RuntimeError(f"unexpected response shape (no 'choices') from "
                                   f"{url}: {str(data)[:300]}")
            return data["choices"][0]["message"]
        except requests.RequestException as e:
            last_err = f"{type(e).__name__}: {e}  (endpoint {url})"
            time.sleep(2 ** attempt)
    raise RuntimeError(f"LLM call failed after 3 attempts: {last_err}")

def llm_ping():
    """connectivity + authentication test — run this once before chatting"""
    try:
        m = llm_chat([{"role": "user",
                       "content": "reply with the single word: pong"}],
                     temperature=0.0, max_tokens=8)
        print(f"[llm] LLM reachable     : {LLM_MODEL} @ {LLM_BASE_URL}")
        print(f"[llm] key authenticates : yes ({_key_state()})")
        print(f"[llm] reply             : {m.get('content')!r}")
        return True
    except RuntimeError as e:
        print("[llm] LLM NOT ready — fix the key/endpoint, then re-run this cell.\n")
        print(f"[llm] {e}")
        return False

def _col_properties():
    """tool schema properties, derived from the bundles (units + typical range)."""
    props = {}
    for c in COND_COLS:
        mu, sd = TH_MU[c], TH_SD[c]
        rng_txt = f"typical {mu - 3 * sd:.3g} to {mu + 3 * sd:.3g}"
        props[c] = dict(type='number',
                        description=f"{UNITS.get(c, '')}; {rng_txt};"
                        + (" wall temperature - two supported islands, see description"
                           if c == WALL_COL else
                           " leave unset to auto-fill from the nearest training row"))
    return props

_isl = '; '.join(f"{k} {v[0]:g}-{v[1]:g} degC" for k, v in ISLANDS.items() if v)
TOOL_DEFS = [
    dict(type='function', function=dict(
        name='boiling_render',
        description=("Render one boiling frame for a descriptor ask. Runs BOTH models "
                     "in one call: M4 (CNN generator) renders a frame, M2 (retriever) "
                     "returns the nearest REAL simulated frame, and the response merges "
                     "them per column as a range. Use for 'what does it look like', "
                     "sensitivity (call twice: baseline + modified) and "
                     "nearest-real-case questions."),
        parameters=dict(type='object', properties=_col_properties()))),
    dict(type='function', function=dict(
        name='list_conditions',
        description=("List the simulations in the corpus: families, wall "
                     "temperatures, frame counts. Use for 'what data/cases exist'."),
        parameters=dict(type='object', properties={}))),
    dict(type='function', function=dict(
        name='boiling_forecast',
        description=("Forecast the descriptor trajectory with the TCN from a real "
                     "window of a simulation. Point trajectory up to 20 steps "
                     "(2 s); beyond that, statistics (onset time, burst period). "
                     "Use for 'what happens next' / 'when does the first bubble "
                     "appear'."),
        parameters=dict(type='object', properties=dict(
            wallTemp=dict(type='number', description='pick the simulation nearest '
                         'this wall temperature (degC)'),
            stem=dict(type='string', description='exact simulation stem, '
                      'e.g. from list_conditions'),
            horizon=dict(type='number', description='steps ahead, 1-80 (0.1 s each)'),
            origin=dict(type='string', enum=['end', 'onset'],
                        description="'end' = latest state (fast), 'onset' = just "
                                    "before the first bubble appears (streams the "
                                    "whole file once, ~1 min, cached)"))))),
]

def _routing_block():
    """INTENT_MAP rendered as the prompt's routing table."""
    lines = ['QUESTION ROUTING - classify the question first, then follow its row:']
    for r in INTENT_MAP:
        tool = r['tool'] or 'NO TOOL - answer directly from the model card'
        lines.append(f"- {r['intent'].upper()} -> {tool}")
        lines.append(f"  process: {r['process']}")
        lines.append('  the answer must state: ' + '; '.join(r['must_state']))
    lines.append('Any other state question is a RENDER question (default row).')
    return '\n'.join(lines)

def build_system_prompt():
    c = MODEL_CARD
    return f"""You are the assistant of a boiling-simulation surrogate service. You answer
questions about near-wall boiling states by calling tools - never from memory -
and you merge the two models into one honest answer.

{c['framing']}

The corpus: 15 pool-boiling simulations (two disjoint wall-temperature islands:
{_isl}); file-level train/val/test splits, so held-out conditions were never
trained on. Descriptor: 5 wall-normal regions (region 1 = at the wall) with void
fraction alpha, mean temperature Tmean and hot fraction f_hot, plus Sauter
diameter D32 (px) and bubble count nbub.

TOOLS
- boiling_render: M4 (CNN, test round-trip {c['m4']['test_rt_pct']}%, CAL
  {c['m4']['test_rt_cal_pct']}%) renders a frame AND M2 (retriever, test
  {c['m2']['test_rt_pct']}%) returns the nearest REAL frame; the response merges
  both per column as a range with an agreement verdict. Unspecified columns are
  auto-filled from the nearest training row; |z| is clamped at 2.5 and reported.
- list_conditions: the corpus inventory.
- boiling_forecast: the TCN ({c['tcn']['test_h20_pct']}% at H=20 vs persistence
  {c['tcn']['persistence_test_h20_pct']}%) rolls descriptors forward;
  {c['horizon_policy']}

{_routing_block()}

RELIABILITY FACTS (use them, including for direct no-tool answers)
- floors: real-next {c['floors']['real_next_test_pct']}% (test), chaos floor
  {c['floors']['chaos_floor_h80_pct']}% at H=80; M4 test RT {c['m4']['test_rt_pct']}%
  (median {c['m4']['median']}%, p90 {c['m4']['p90']}%, {c['m4']['blowups']} blow-ups),
  M2 {c['m2']['test_rt_pct']}%.
- per column: Tmean {c['m4']['per_col_pct']['Tmean']}% and f_hot_1
  {c['m4']['per_col_pct']['f_hot_1']}% (reliable); alpha_2-5
  {c['m4']['per_col_pct']['alpha_2_5']}%; alpha_1 {c['m4']['per_col_pct']['alpha_1']}%,
  nbub {c['m4']['per_col_pct']['nbub']}%, D32 {c['m4']['per_col_pct']['D32']}% (not).
- knobs: wallTemp gain 0.01 (dead), D32 gain -0.45 (wrong sign), alpha knobs
  cross-leak ~1 (moving one alpha moves the others).
- forecasting: onset MAE {c['tcn']['onset_mae_steps']} steps vs burst period
  {c['tcn']['burst_period_steps']}; event F1 = 0 (do not promise individual events).

HOW TO ANSWER
1. Classify the question with the routing table; follow that row's process and
   must-state list. Call the tool BEFORE answering (except reliability questions,
   which the card above already answers).
2. Quote the tool's numbers (round-trip %, NNgap in sigma, merged ranges). The
   merged range IS the answer - never a single model's number.
3. Off-manifold wall temperatures (between the islands) must be flagged; suggest
   the nearest supportable ask instead of pretending.
4. For forecasts beyond 20 steps give statistics only; never a point void
   trajectory. For sensitivity questions, call boiling_render twice (baseline
   and modified ask) and report which columns moved.
5. If asked for anything in this list, say it is not available and offer the
   closest tool-based alternative: {', '.join(c['not_available'])}.
6. NEVER format your answer as a table (no | pipes, no aligned columns, no
   markdown headers): the web page already renders the merged table, both
   images and a deterministic "In short" box under your text. Answer in 3-6
   short prose sentences.
Keep answers short, concrete, with the numbers from the tool response."""

# ---- the call used by the loop (placeholder LLM when no key is set) -----------
def call_llm(messages, tools=TOOL_DEFS):
    """-> (content str, tool_calls list | None); llm_chat when a real key is
    set, the scripted placeholder otherwise (same routing table either way)."""
    if not _key_is_real():
        return fake_llm_turn(messages, tools)
    m = llm_chat(messages, tools=tools if tools else None)
    return (m.get('content') or '', m.get('tool_calls') or None)

# ---- scripted placeholder LLM (offline / fixture mode) ------------------------
def _parse_wt(user):
    m = re.search(r'(-?\d+(?:\.\d+)?)\s*(?:°c|deg(?:rees)?\s*c\b|c(?![a-z]))',
                  user.lower())
    return float(m.group(1)) if m else None

def _default_wt(low):
    wt = _parse_wt(low)
    if wt is not None:
        return wt
    if any(w in low for w in ('boil', 'hot', 'nucleat', 'bubble')):
        return float(np.mean(_wts_bo)) if _wts_bo else 85.0
    return float(np.mean(_wts_sb)) if _wts_sb else 20.0

def _fake_tool_calls(user):
    """route the question with INTENT_MAP and build the matching tool call(s)."""
    low = user.lower()
    row = route_intent(user)
    wt = _default_wt(low)
    if row['intent'] == 'forecast':
        origin = 'onset' if any(w in low for w in ('first bubble', 'appear',
                                                   'onset', 'nucleat')) else 'end'
        horizon = 60 if any(w in low for w in ('long', 'period', 'plateau')) else 20
        return [dict(id='call_fake0', type='function',
                     function=dict(name='boiling_forecast',
                                   arguments=json.dumps(dict(
                                       wallTemp=wt, horizon=horizon,
                                       origin=origin))))], \
            f"Routing: {row['intent']} -> boiling_forecast (wallTemp={wt:g}, " \
            f"origin={origin}, horizon={horizon})."
    if row['intent'] == 'conditions':
        return [dict(id='call_fake0', type='function',
                     function=dict(name='list_conditions', arguments='{}'))], \
            f"Routing: {row['intent']} -> list_conditions."
    if row['intent'] == 'sensitivity':
        base = dict(wallTemp=wt)
        mod = dict(wallTemp=wt, alpha_1=0.25)      # a concrete steered variant
        return [dict(id='call_fake0', type='function',
                     function=dict(name='boiling_render',
                                   arguments=json.dumps(base))),
                dict(id='call_fake1', type='function',
                     function=dict(name='boiling_render',
                                   arguments=json.dumps(mod)))], \
            f"Routing: {row['intent']} -> two boiling_render calls " \
            f"(baseline, then alpha_1=0.25)."
    return [dict(id='call_fake0', type='function',
                 function=dict(name='boiling_render',
                               arguments=json.dumps(dict(wallTemp=wt))))], \
        f"Routing: {row['intent']} -> boiling_render (wallTemp={wt:g})."

def fake_llm_turn(messages, tools):
    """deterministic stand-in routing with the SAME intent table as the prompt:
    one tool round (or a direct card answer), then a template answer built from
    the actual tool result - proves the loop wiring without any network."""
    user = next((m['content'] for m in reversed(messages) if m['role'] == 'user'), '')
    row = route_intent(user)
    has_result = any(m['role'] == 'tool' for m in messages)
    if not has_result:
        if row['intent'] in ('reliability', 'not_available'):
            if row['intent'] == 'not_available':
                _c2 = next(r for r in INTENT_MAP if r['intent'] == 'not_available')
                return (f"This is not available: the surrogates cover only the 13 "
                        f"descriptor columns, wall temperature and short-term "
                        f"forecasting. {_c2['process']}. "
                        f"[offline placeholder answer]"), None
            c = MODEL_CARD
            return (f"From the model card, no tool needed: M4 test round-trip "
                    f"{c['m4']['test_rt_pct']}% (CAL {c['m4']['test_rt_cal_pct']}%), "
                    f"M2 {c['m2']['test_rt_pct']}%, floors real-next "
                    f"{c['floors']['real_next_test_pct']}% and chaos "
                    f"{c['floors']['chaos_floor_h80_pct']}% at H=80. Per column: "
                    f"Tmean {c['m4']['per_col_pct']['Tmean']}%, f_hot_1 "
                    f"{c['m4']['per_col_pct']['f_hot_1']}%, D32 "
                    f"{c['m4']['per_col_pct']['D32']}%, nbub "
                    f"{c['m4']['per_col_pct']['nbub']}%. The wall-temperature knob "
                    f"is dead (gain 0.01). [offline placeholder answer]"), None
        tcs, plan = _fake_tool_calls(user)
        return plan, tcs
    # ---- answer from the LAST tool result (deterministic template) ------------
    tr = json.loads(next(m['content'] for m in reversed(messages)
                         if m['role'] == 'tool'))
    if tr.get('ok') is False:
        return f"The tool could not answer: {tr.get('reason')}. [offline placeholder]", None
    if tr.get('tool') == 'boiling_render':
        ans = (f"Here are both temperature frames. The generated frame (M4) matches "
               f"the ask with round-trip {tr['m4']['round_trip_pct']}%; the nearest "
               f"real frame is {tr['m2']['stem']} at t = {tr['m2']['t_s']} s "
               f"(distance {tr['m2']['distance_sigma']} sigma). "
               f"{tr['merged']['summary'].capitalize()}. ")
    elif tr.get('tool') == 'boiling_forecast':
        s = tr.get('stats', {})
        ans = (f"TCN forecast from {s.get('stem')} over {s.get('horizon_steps')} "
               f"steps (origin {s.get('origin_effective', s.get('origin'))}): onset "
               f"at step {s.get('onset_step')}, {s.get('n_burst_episodes')} burst "
               f"episodes. Point values stop at step 20; beyond that, statistics "
               f"only. ")
    else:
        ans = f"The corpus holds {len(tr.get('rows', []))} simulations. "
    return (ans + '[offline placeholder answer - set LLM_API_KEY in .env for real '
            'Gemini answers]'), None

# ---- the agentic loop ----------------------------------------------------------
def dispatch_tool(name, args):
    """tool schemas are flat; the render tool takes them as one ask dict."""
    if name == 'boiling_render':
        return tool_boiling_render(args or {})
    if name == 'boiling_forecast':
        return tool_boiling_forecast(**(args or {}))
    if name == 'list_conditions':
        return tool_list_conditions()
    return dict(tool=name, ok=False, reason=f'unknown tool {name!r}')

SESSIONS = {}
AUDIT_PATH = os.path.join(RUNTIME_DIR, 'audit.jsonl')
PROGRESS = {}          # session_id -> live 'thinking' state polled by the page

def _prog_reset(sid):
    """fresh live state for a chat turn; keeps the registry bounded."""
    if len(PROGRESS) > 80:                    # finished sessions are dropped first
        for k in [k for k, v in PROGRESS.items() if v.get('done')][:40]:
            PROGRESS.pop(k, None)
    PROGRESS[sid] = dict(steps=[], done=False, error=None)

def _trim_history(history, keep=40):
    """cap the context window WITHOUT splitting an assistant-tool_calls message
    from its tool results: cut back to the last user-message boundary."""
    if len(history) <= keep:
        return history
    cut = max(0, len(history) - keep)
    for i in range(cut, len(history)):
        if history[i].get('role') == 'user':
            return history[i:]
    return history[cut:]

def _fn_args(tc):
    """OpenAI uses 'arguments' (string); some compatible servers use 'args'."""
    fn = tc.get("function", {})
    raw = fn.get("arguments") or fn.get("args") or "{}"
    try:
        return json.loads(raw) if isinstance(raw, str) else dict(raw)
    except json.JSONDecodeError:
        return {}

def _strip_tables(text):
    """remove markdown-style table lines from the final answer: the page renders
    real tables plus the deterministic 'In short' box, so a pipe table in the
    answer text would appear as raw text on top of the summary and duplicate it.
    Conservative: a line must look like a table row (>=2 pipes, pipe at an edge,
    or a pure |---| separator) to be dropped; prose is never touched."""
    if not text or '|' not in text:
        return text
    kept = []
    for ln in text.splitlines():
        s = ln.strip()
        if s.count('|') >= 2 and (s.startswith('|') or s.endswith('|')):
            continue                                    # | a | b | row
        if s and set(s) <= set('|-: '):                 # |---|---| / ---|---
            continue
        kept.append(ln)
    return '\n'.join(kept).strip()

def run_chat(user_message, session_id='default', max_rounds=4):
    events, renders, forecasts, lists_ = [], [], [], []

    def ev(kind, text):
        events.append(dict(step=len(events) + 1, kind=kind, text=str(text)))
        _p = PROGRESS.get(session_id)         # live mirror for the thinking UI
        if _p is not None and not _p['done']:
            _p['steps'].append((str(kind), str(text)))

    _prog_reset(session_id)
    history = _trim_history(SESSIONS.get(session_id) or [])
    if not history or history[0].get('role') != 'system':
        history.insert(0, dict(role='system', content=build_system_prompt()))
    history.append(dict(role='user', content=user_message))
    SESSIONS[session_id] = history
    ev('understand', f"received: {user_message[:90]}"
       + ('...' if len(user_message) > 90 else ''))
    answer = ''
    try:
        for _round in range(max_rounds):
            ev('think', f"round {_round + 1}/{max_rounds}: asking "
                        f"{'the LLM' if _key_is_real() else 'the offline router'}")
            try:
                content, tool_calls = call_llm(history)
            except Exception as e:
                answer = f'[LLM unavailable: {e}]'
                ev('answer', 'LLM error - answered with the stored error message')
                break
            if content:
                ev('plan', content[:220])
            if not tool_calls:
                answer = content or ''
                ev('answer', f'final answer written ({len(answer.split())} words)')
                break
            history.append(dict(role='assistant', content=content or '',
                                tool_calls=tool_calls))
            for tc in tool_calls:
                name = tc['function']['name']
                args = _fn_args(tc)
                ev('call', f"calling {name}({', '.join(f'{k}={v}' for k, v in
                                                       args.items())})")
                try:
                    result = dispatch_tool(name, args)
                except Exception as e:
                    import traceback
                    result = dict(tool=name, ok=False,
                                  reason=f'tool error: {e}',
                                  _tb=traceback.format_exc()[-600:])
                for line in condense(result):
                    ev('tool', line)
                if result.get('_tb'):
                    print(result['_tb'])
                    result.pop('_tb', None)
                if result.get('tool') == 'boiling_render':
                    renders.append(result)
                elif result.get('tool') == 'boiling_forecast':
                    forecasts.append(result)
                else:
                    lists_.append(result)
                history.append(dict(role='tool', tool_call_id=tc.get('id', 'call'),
                                    content=json.dumps(result, default=str)))
        else:
            # round budget spent: force a final answer WITHOUT tools
            ev('think', 'tool round budget spent - forcing a direct answer')
            try:
                content, _ = call_llm(history, tools=None)
                answer = content or answer
            except Exception as e:
                answer = answer or f'[LLM unavailable: {e}]'
            ev('answer', 'final answer after the tool round limit')
    finally:
        _p = PROGRESS.get(session_id)
        if _p is not None:
            _p['done'] = True
            _p['error'] = (answer if answer.startswith('[LLM unavailable')
                           else None)
    SESSIONS[session_id] = _trim_history(history)
    answer = _strip_tables(answer)   # no text tables above the 'In short' box
    with open(AUDIT_PATH, 'a', encoding='utf-8') as fh:
        fh.write(json.dumps(dict(ts=time.strftime('%Y-%m-%d %H:%M:%S'),
                                 session=session_id, message=user_message,
                                 events=events,
                                 tools=[r.get('tool') for r in renders + forecasts],
                                 answer=answer), default=str) + '\n')
    return dict(answer=answer, events=events, renders=renders,
                forecasts=forecasts, lists=lists_, session_id=session_id,
                summary=reply_summary(renders, forecasts, lists_))

# ---- smoke tests (offline if the real LLM is unreachable) ---------------------
def _force_offline(fn):
    """run fn with the LLM temporarily replaced by the offline placeholder."""
    global call_llm
    _saved = call_llm
    call_llm = lambda m, t=None: fake_llm_turn(m, t)
    try:
        return fn()
    finally:
        call_llm = _saved

if _key_is_real():
    llm_ping()                                  # connectivity + auth check
else:
    print(f"[llm] no real key ({_key_state()}) — offline placeholder router active; "
          f"tools and pages work, chat answers are scripted templates")

# the answer must arrive as prose: tables live in the page's cards + 'In short'
_st = _strip_tables('prose line\n| column | M4 | M2 |\n|---|---|---|\n'
                    '| nbub | 12 | 14 |\n'
                    'a | b | c without edge pipes\nstill prose')
assert _st == 'prose line\na | b | c without edge pipes\nstill prose', \
    f'_strip_tables mangled: {_st!r}'
assert 'NEVER format your answer as a table' in build_system_prompt(), \
    'the prompt must forbid text tables in the answer'

_r = run_chat('what does the wall region look like at 21 degrees C?', 'smoke')
if _r['answer'].startswith('[LLM unavailable') or \
        not (_r['renders'] or _r['lists'] or _r['forecasts']):
    print('[llm] real LLM unreachable or called no tool in the smoke - '
          're-running the loop offline to verify the wiring')
    _r = _force_offline(lambda: run_chat(
        'what does the wall region look like at 21 degrees C?', 'smoke_off'))
_r2 = _force_offline(lambda: run_chat('how reliable is D32?', 'smoke_card'))
assert len(_r['events']) >= 4 and (_r['renders'] or _r['lists'] or _r['forecasts']), \
    f'agentic loop smoke failed: {json.dumps(_r["events"])[:300]}'
assert _r2['renders'] == [] and _r2['forecasts'] == [] and 'model card' in _r2['answer'], \
    'reliability route did not answer directly from the card'
print(f"[llm] loop smoke ({'offline placeholder' if not LLM_API_KEY else LLM_MODEL}): "
      f"{len(_r['events'])} trace steps, "
      f"tools called: {sorted({e['text'].split('(')[0].replace('calling ', '') for e in _r['events'] if e['kind'] == 'call'})}")
print('[llm] reliability route: direct model-card answer, no tool call PASS')
print(f"[llm] audit log -> {AUDIT_PATH}")
print('[ok] CELL 7 done — OpenAI-compatible client + routing + agentic loop live '
      '(every step published to PROGRESS for the thinking indicator)')


[llm] LLM reachable     : gemini-3.1-flash-lite @ https://generativelanguage.googleapis.com/v1beta/openai/
[llm] key authenticates : yes (set, 53 characters, starts with 'AQ.A')
[llm] reply             : 'pong'
[llm] loop smoke (gemini-3.1-flash-lite): 10 trace steps, tools called: ['boiling_render']
[llm] reliability route: direct model-card answer, no tool call PASS
[llm] audit log -> G:\BoilDesc\boildesc_runtime\audit.jsonl
[ok] CELL 7 done — OpenAI-compatible client + routing + agentic loop live (every step published to PROGRESS for the thinking indicator)


## 8 · The service and the browser page

FastAPI endpoints: `POST /chat` (the LLM loop), `GET /progress` (the live step
registry of the running chat — the thinking indicator's data source),
`GET /schema` (option space and the routing table), `GET /card`,
`POST /tools/render` and `/tools/forecast` (direct tool calls, no LLM — the
manual/offline mode), `/static` for tool-written images. The page is a single
self-contained HTML file: a header strip explaining **all four question routes**
(render / forecast / data / trust) with an example question and what each
returns, a **live thinking bubble** — a spinner that polls `/progress` every
700 ms and shows "thinking · step N: …" plus elapsed seconds while the chat is
still running, so slow LLM rounds and the ~1-minute onset-origin forecast show
progress instead of silence — a numbered thinking-steps timeline
(service-generated, data-driven), and result cards designed to be read at a
glance: a headline row with **stat chips** (round-trip %, retrieval gap σ,
agreement count, onset/burst), the **two temperature frames side by side on one
shared color scale** (the range in each caption), images that are never cropped
(`object-fit: contain`, and the figures get explicit layout padding so no title
or colorbar can touch the canvas edge — a clipped title looks exactly like a
cut-off image), the merged table with units inline, zebra rows and color-coded
verdicts, flags, caveats and the floor line. The LLM's answer arrives **as prose
only**: the prompt forbids tables and a conservative filter strips markdown
table lines, so no raw text table can appear above the summary and duplicate
it. Every answer ends with the **"In short"** box: deterministic bullet lines
that explain the results. The suggestion buttons and the input field form one
**sticky bottom bar** — the buttons ride on top of the field and stay visible
on the empty page, through long answers, and while scrolling. The cell ends
with an end-to-end test that walks **every routing row** — render, forecast,
conditions, reliability (no tools), sensitivity (two renders) — verifying
`/progress` reports the finished chat, that no saved image has a single dark
pixel on its border (the no-cropping guarantee), that both heat maps share one
scale, that the reply summary is present and the answer carries no table lines,
and that the sticky question bar is in the page, falling back to the offline
placeholder if the real LLM is unreachable, so the cell cannot fail merely
because the key or the network is down.

In [8]:
# CELL 8 · the FastAPI service + the browser page + end-to-end test of every route
#
# Endpoints: POST /chat (LLM loop), GET /schema (option space), GET /card,
# POST /tools/render and /tools/forecast (direct tool calls, no LLM — also the
# manual/offline mode), GET / (the page below), /static (tool-written images).
#
# v2: the header explains ALL four question routes (render / forecast / data /
# trust); the e2e test walks every routing row — render, forecast, conditions,
# reliability (direct card answer, no tool) and sensitivity (two renders) — and
# falls back to the offline placeholder LLM if the real LLM is unreachable, so
# the cell cannot fail just because the key or the network is down.
# v4: GET /progress exposes the loop's live step registry, and the page polls it
# while /chat is in flight — the user sees "thinking · step N: …" instead of a
# silent wait.
from fastapi import FastAPI
from fastapi.staticfiles import StaticFiles
from fastapi.responses import HTMLResponse
from pydantic import BaseModel

class ChatIn(BaseModel):
    message: str
    session_id: str = 'default'

class RenderIn(BaseModel):
    ask: dict = {}

class ForecastIn(BaseModel):
    wallTemp: float | None = None
    stem: str | None = None
    horizon: int = 20
    origin: str = 'end'

app = FastAPI(title='BoilDesc surrogate service')
app.mount('/static', StaticFiles(directory=os.path.join(RUNTIME_DIR, 'static')),
          name='static')

@app.post('/chat')
def chat_endpoint(inp: ChatIn):
    return run_chat(inp.message, inp.session_id or 'default')

@app.get('/progress')
def progress_endpoint(sid: str = ''):
    """live 'thinking' state for the page's spinner: which step the chat is on.
    The agentic loop mirrors every trace step into PROGRESS[session_id] while
    /chat is still running, so the browser can show progress, not silence."""
    p = PROGRESS.get(sid or '')
    if not p:
        return dict(steps=[], done=False, error=None, n=0, unknown=True)
    return dict(steps=[dict(step=i + 1, kind=k, text=t)
                       for i, (k, t) in enumerate(p['steps'])],
                done=p['done'], error=p['error'], n=len(p['steps']))

@app.get('/schema')
def schema_endpoint():
    return dict(islands={k: list(v) for k, v in ISLANDS.items() if v},
                gap=list(GAP) if GAP else None,
                columns=[dict(name=c, unit=UNITS.get(c, ''), mu=round(TH_MU[c], 3),
                              sd=round(TH_SD[c], 4)) for c in COND_COLS],
                live=LIVE_COLS, fixture=FIXTURE,
                llm_model=LLM_MODEL, llm_key_set=bool(LLM_API_KEY),
                forecast_available=FC_AVAILABLE,
                routing=[dict(intent=r['intent'], tool=r['tool'],
                              process=r['process']) for r in INTENT_MAP])

@app.get('/card')
def card_endpoint():
    return MODEL_CARD

@app.post('/tools/render')
def render_endpoint(inp: RenderIn):
    return tool_boiling_render(inp.ask)

@app.post('/tools/forecast')
def forecast_endpoint(inp: ForecastIn):
    return tool_boiling_forecast(wallTemp=inp.wallTemp, stem=inp.stem,
                                 horizon=inp.horizon, origin=inp.origin)

_ISL = '; '.join(f"{k} {v[0]:g}-{v[1]:g} °C" for k, v in ISLANDS.items() if v)
PAGE = """<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>BoilDesc — boiling surrogate</title>
<style>
:root{--bg:#0f1419;--card:#171f27;--ink:#e8edf2;--dim:#9aa7b0;--line:#2a3641;
--accent:#3557a0;--ok:#2e8b57;--warn:#b8860b;--bad:#c0563e}
*{box-sizing:border-box}
body{margin:0;background:var(--bg);color:var(--ink);
font-family:system-ui,-apple-system,'Segoe UI',Roboto,sans-serif;
font-size:clamp(14px,1.6vw,16px)}
.wrap{max-width:960px;margin:0 auto;padding:clamp(10px,2vw,22px)}
header{border-bottom:1px solid var(--line);padding-bottom:14px;margin-bottom:14px}
h1{font-size:clamp(19px,3.4vw,26px);margin:0 0 4px}
.sub{color:var(--dim);font-size:clamp(12px,1.5vw,14px);margin-bottom:12px}
.how{display:grid;grid-template-columns:repeat(auto-fit,minmax(220px,1fr));gap:10px}
.howcard{background:var(--card);border:1px solid var(--line);border-radius:10px;
padding:10px 12px;font-size:clamp(11px,1.4vw,13px)}
.howcard b{display:block;font-size:clamp(12px,1.6vw,14px);margin-bottom:4px}
.howcard .q{color:var(--ink);font-style:italic}
.howcard .t{color:#7fa3e0;margin:5px 0}
.flow{margin:10px 0 0;color:var(--dim);font-size:clamp(11px,1.4vw,13px);text-align:center}
.banner{background:#3a2f16;border:1px solid #6b5a1e;color:#e6d9a8;border-radius:8px;
padding:8px 12px;margin:12px 0;font-size:clamp(11px,1.4vw,13px);display:none}
#chat{display:flex;flex-direction:column;gap:14px;min-height:200px}
.msg{border-radius:12px;padding:10px 14px;max-width:88%}
.msg.user{align-self:flex-end;background:#24406e;border:1px solid #35557f}
.msg.bot{align-self:stretch;background:var(--card);border:1px solid var(--line)}
.trace{margin:0 0 10px;padding:0;list-style:none}
.trace li{display:flex;gap:8px;padding:3px 0;color:var(--dim);
font-size:clamp(11px,1.5vw,13px);opacity:0;animation:up .4s ease forwards;
animation-delay:calc(var(--i)*.12s)}
@keyframes up{from{opacity:0;transform:translateY(6px)}to{opacity:1;transform:none}}
.trace .n{color:#5f7a99;min-width:18px;text-align:right}
.trace .k{min-width:52px;text-transform:uppercase;font-size:10px;letter-spacing:.05em}
.k-understand,.k-plan,.k-think{color:#7fa3e0}.k-call{color:#c9a227}.k-tool{color:var(--ok)}
.k-answer{color:var(--ink)}
.think{display:flex;align-items:center;gap:10px;color:var(--dim);
font-size:clamp(12px,1.5vw,14px);padding:4px 2px}
.spin{flex:none;width:13px;height:13px;border:2px solid var(--line);
border-top-color:#7fa3e0;border-radius:50%;animation:rot 1s linear infinite}
@keyframes rot{to{transform:rotate(360deg)}}
.think .tk{flex:none;min-width:52px;text-transform:uppercase;font-size:10px;
letter-spacing:.05em;color:#7fa3e0}
.think .tt{flex:1;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
.images{display:grid;grid-template-columns:1fr 1fr;gap:10px;margin:10px 0}
.images img{width:100%;height:auto;object-fit:contain;border-radius:8px;
border:1px solid var(--line)}   /* never crop: the whole PNG stays visible */
.images figcaption{color:var(--dim);font-size:11px;text-align:center;padding-top:4px;
line-height:1.45}
.fig{width:100%;height:auto;object-fit:contain;border-radius:8px;
border:1px solid var(--line);margin:8px 0}
.tablewrap{overflow-x:auto;margin:8px 0}
table{border-collapse:collapse;width:100%;font-size:clamp(11px,1.5vw,13px)}
th,td{border:1px solid var(--line);padding:4px 8px;text-align:right;white-space:nowrap}
th{background:#1d2831;color:var(--dim);font-weight:600}
tbody tr:nth-child(even){background:#131b22}
td:first-child,th:first-child{text-align:left}
.unit{color:var(--dim);font-size:10px}
.statline{display:flex;flex-wrap:wrap;gap:6px;margin:2px 0 8px}
.stat{background:#1a242e;border:1px solid var(--line);border-radius:6px;
padding:2px 9px;font-size:clamp(11px,1.4vw,12.5px);color:#bcd0e0}
.stat i{font-style:normal;color:var(--dim);margin-right:5px;
font-size:10px;text-transform:uppercase;letter-spacing:.04em}
.cardhead{display:flex;flex-wrap:wrap;gap:6px;align-items:baseline;justify-content:
space-between;border-bottom:1px solid var(--line);padding-bottom:6px;margin-bottom:6px}
.cardhead b{color:#7fa3e0;font-size:clamp(12px,1.6vw,14px);
letter-spacing:.02em}
.sumbox{margin-top:10px;background:#1a242e;border:1px solid var(--accent);
border-left:4px solid var(--accent);border-radius:8px;padding:8px 12px}
.sumbox b{display:block;color:#7fa3e0;font-size:11px;text-transform:uppercase;
letter-spacing:.05em;margin-bottom:4px}
.sumbox ul{margin:0;padding-left:16px}
.sumbox li{margin:3px 0;line-height:1.45;font-size:clamp(12px,1.5vw,14px)}
.v-ok{color:var(--ok)}.v-partial{color:var(--warn)}.v-disputed{color:var(--bad)}
.chips{display:flex;flex-wrap:wrap;gap:6px}
.chip{background:#21303c;border:1px solid var(--line);border-radius:20px;
padding:3px 10px;font-size:clamp(11px,1.4vw,12.5px);color:#bcd0e0;cursor:pointer}
.chip:hover{border-color:var(--accent)}
.cav{background:#2b2320;border:1px solid #5a3d33;color:#d9b8a6;border-radius:6px;
padding:2px 8px;font-size:11px}
.ans{border-top:1px dashed var(--line);padding-top:8px;line-height:1.5}
.stats td:first-child{color:var(--dim)}
/* sticky question bar: the suggestion buttons ride ON TOP of the input field
and stay visible whether the page is empty, full of answers, or scrolled */
.foot{position:sticky;bottom:0;z-index:5;background:var(--bg);margin-top:16px;
padding:8px 0 2px;border-top:1px solid var(--line)}
form{display:flex;gap:8px;padding:8px 0 0}
input[type=text]{flex:1;background:#141c23;border:1px solid var(--line);
border-radius:10px;padding:10px 14px;color:var(--ink);font-size:inherit}
button{background:var(--accent);color:#fff;border:0;border-radius:10px;
padding:10px 18px;font-size:inherit;cursor:pointer}
button:disabled{opacity:.5}
@media(max-width:640px){.images{grid-template-columns:1fr}}
</style></head><body><div class="wrap">
<header>
<h1>BoilDesc — boiling surrogate, two models per answer</h1>
<div class="sub">__ISLANDS__&nbsp;·&nbsp;__MODE__</div>
<div class="how">
<div class="howcard"><b>Render a state</b>
<span class="q">"what does the wall region look like at 21 °C?"</span>
<div class="t">→ boiling_render</div>
M4 (CNN) generates a frame <em>and</em> M2 retrieves the nearest real simulated
frame; both are re-analysed → <b>two temperature frames + a merged table</b></div>
<div class="howcard"><b>Look ahead</b>
<span class="q">"what happens in the next 2 s? when is the first bubble?"</span>
<div class="t">→ boiling_forecast</div>
the TCN rolls the 13 descriptors forward step by step (own predictions fed back)
→ <b>trajectory chart + onset / burst statistics</b></div>
<div class="howcard"><b>Inspect the data</b>
<span class="q">"which cases are in the corpus?"</span>
<div class="t">→ list_conditions</div>
scans the simulations: family, wall temperature, frames → <b>conditions table</b></div>
<div class="howcard"><b>Trust &amp; limits</b>
<span class="q">"how reliable is D32? what can't it do?"</span>
<div class="t">→ model card, no tool</div>
answered directly from the validated facts: floors, per-column errors, dead
knobs, the chaos limit → <b>honest reliability answer</b></div>
</div>
<div class="flow">you ask → Gemini classifies the question → the routing row's tool
runs → the answer merges both models as a range with confidence</div>
</header>
<div class="banner" id="banner">offline placeholder mode — no API key set. Tools and
pages work; chat answers are scripted templates routed the same way. Add
LLM_API_KEY to .env and re-run.</div>
<div id="chat"></div>
<div class="foot">
<div class="chips" id="chips"></div>
<form id="f" onsubmit="return go(event)">
<input type="text" id="q" placeholder="ask about a boiling state, a forecast, the corpus, or reliability..."
autocomplete="off"><button id="b">Send</button></form>
</div>
</div>
<script>
const SID = 's' + Math.random().toString(36).slice(2, 9);
const chips = ["what does the wall region look like at 21 °C?",
 "show me boiling at 100 °C with a larger void fraction",
 "what if the void fraction doubles?",
 "what happens in the next 2 seconds at 21 °C?",
 "when does the first bubble appear?",
 "which cases are in the data?",
 "how reliable is D32?"];
const esc = s => String(s ?? '').replace(/[&<>"']/g,
 c => ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]));
function chipRow(){ const d = document.getElementById('chips');
 d.innerHTML = ''; chips.forEach(c => { const b = document.createElement('button');
 b.type = 'button'; b.className = 'chip'; b.textContent = c;
 b.onclick = () => { document.getElementById('q').value = c; go(); }; d.appendChild(b); }); }
function userMsg(t){ const m = document.createElement('div');
 m.className = 'msg user'; m.textContent = t;
 document.getElementById('chat').appendChild(m); }
const rg = a => Array.isArray(a) ? `${a[0]} – ${a[1]}` : '';   // range pair
function renderCard(r){ const im = r.images || {}, M = r.m4 || {}, M2 = r.m2 || {},
 mg = r.merged || {}, sc = im.shared_scale === true;
 let h = `<div class="cardhead"><b>boiling_render — two models, one answer</b>
 <span class="sub">${esc(r.wall_status || '')}</span></div>`;
 h += '<div class="statline">'
  + `<span class="stat"><i>M4 round-trip</i>${M.round_trip_pct ?? '?'}%</span>`
  + `<span class="stat"><i>retrieval gap</i>${r.nn_sigma ?? '?'} σ</span>`
  + `<span class="stat"><i>agree</i>${(mg.counts || {}).agree ?? '?'}/`
  + `${(mg.rows || []).length}</span>`
  + `<span class="stat"><i>disputed</i>${(mg.counts || {}).disputed ?? 0}</span>`
  + (sc ? '<span class="stat"><i>heat maps</i>one shared color scale</span>' : '')
  + '</div>';
 h += '<div class="images">';
 h += `<figure><img src="${im.m4}" alt="M4 generated temperature frame">
 <figcaption>M4 · generated (round-trip ${M.round_trip_pct ?? '?'}%)`
 + (im.m4_range ? ` · shared scale ${rg(im.m4_range)} °C` : '') +
 `</figcaption></figure>`;
 h += `<figure><img src="${im.m2}" alt="M2 nearest real temperature frame">
 <figcaption>M2 · real · ${esc(M2.stem ?? '?')} · t = ${M2.t_s ?? '?'} s`
 + (im.m2_range ? ` · shared scale ${rg(im.m2_range)} °C` : '') +
 `</figcaption></figure></div>`;
 h += `<div class="tablewrap"><table>
 <tr><th>column</th><th>ask</th><th>M4</th><th>M2 real</th><th>merged range</th>
 <th>gap σ</th><th>verdict</th></tr>`;
 (mg.rows || []).forEach(row => { const v = row.verdict;
  h += `<tr><td>${esc(row.column)} <span class="unit">${esc(row.unit)}</span></td>
  <td>${row.ask}</td><td>${row.m4}</td><td>${row.m2}</td>
  <td>${row.range_lo} – ${row.range_hi}</td><td>${row.gap_sigma}</td>
  <td class="v-${v}">${v === 'agree' ? '✓' : v === 'partial' ? '〜' : '✗'} ${v}</td></tr>`; });
 h += '</table></div><div class="chips">';
 (r.flags || []).forEach(f => h += `<span class="cav">${esc(f)}</span>`);
 h += `<span class="cav">${esc(r.floor_line)}</span>`;
 (r.caveats || []).forEach(cv => h += `<span class="cav">${esc(cv)}</span>`);
 h += `</div><div class="sub">${esc(mg.summary || '')} · NNgap ${r.nn_sigma} σ ·
 auto-filled: ${(r.auto_filled || []).length}</div>`;
 return h; }
const FCL = {stem: 'simulation', origin: 'origin requested',
 origin_effective: 'origin used', horizon_steps: 'horizon (steps)',
 horizon_s: 'horizon (s)', dt_s: 'step (s)', onset_step: 'onset (step)',
 onset_s: 'onset (s)', burst_period_steps: 'burst period (steps)',
 n_burst_episodes: 'burst episodes', plateau: 'plateau (2nd half)',
 clamped_entries: 'clamped entries', note: 'note'};
function forecastCard(f){ if (!f.ok) return `<div class="sub">${esc(f.reason)}</div>`;
 const s = f.stats || {};
 let h = `<div class="cardhead"><b>boiling_forecast — TCN rollout</b>
 <span class="sub">${esc(s.stem ?? '')} · origin ${esc(s.origin_effective ?? '')}</span></div>`;
 h += '<div class="statline">'
  + `<span class="stat"><i>horizon</i>${s.horizon_steps} steps = ${s.horizon_s} s</span>`
  + (s.onset_s != null
     ? `<span class="stat"><i>first bubble</i>step ${s.onset_step} = ${s.onset_s} s</span>`
     : '<span class="stat"><i>onset</i>none in horizon</span>')
  + (s.burst_period_steps != null
     ? `<span class="stat"><i>burst period</i>${s.burst_period_steps} steps</span>` : '')
  + '</div>';
 h += `<img class="fig" src="${f.image}" alt="TCN forecast trajectory chart">`;
 h += '<div class="tablewrap"><table class="stats">';
 for (const [k, v] of Object.entries(s)) {
  if (v === null || v === undefined || k === 'origin_pos') continue;
  h += `<tr><td>${esc(FCL[k] || k)}</td><td>${esc(typeof v === 'object'
   ? Object.values(v)[0] : v)}</td></tr>`; }
 h += '</table></div><div class="chips">';
 (f.caveats || []).forEach(cv => h += `<span class="cav">${esc(cv)}</span>`);
 h += `</div><div class="sub">${esc(f.horizon_policy)}</div>`;
 return h; }
function listCard(l){ let h = `<div class="tablewrap"><table><tr><th>simulation</th><th>wallTemp °C</th><th>family</th><th>frames</th><th>in retriever</th></tr>`;
 l.rows.forEach(r => h += `<tr><td>${esc(r.stem)}</td><td>${r.wallTemp}</td>
 <td>${esc(r.family)}</td><td>${esc(r.n_frames)}</td><td>${esc(r.in_retriever_index)}</td></tr>`);
 h += `</table></div><div class="sub">islands: ${esc(l.islands)}</div>`;
 return h; }
function botMsg(d){ const m = document.createElement('div'); m.className = 'msg bot';
 let h = '<ol class="trace">';
 (d.events || []).forEach((e, i) => { h += `<li style="--i:${i}"><span class="n">${i + 1}</span>
 <span class="k k-${e.kind}">${e.kind}</span><span>${esc(e.text)}</span></li>`; });
 h += '</ol>';
 (d.renders || []).forEach(r => h += renderCard(r));
 (d.forecasts || []).forEach(f => h += forecastCard(f));
 (d.lists || []).forEach(l => h += listCard(l));
 h += `<div class="ans">${d.answer ? esc(d.answer) : ''}</div>`;
 if (d.summary && d.summary.length) {          // deterministic "what this means"
  h += '<div class="sumbox"><b>In short</b><ul>'
   + d.summary.map(s => `<li>${esc(s)}</li>`).join('') + '</ul></div>'; }
 m.innerHTML = h; document.getElementById('chat').appendChild(m); }
function thinkingMsg(){ const m = document.createElement('div');
 m.className = 'msg bot'; m.innerHTML =
  '<div class="think"><span class="spin"></span><span class="tk">thinking</span>' +
  '<span class="tt">thinking…</span></div>';
 document.getElementById('chat').appendChild(m); return m; }
function tickThink(m, txt){ const t = m.querySelector('.tt'); if (t) t.textContent = txt; }
async function go(ev){ if (ev) ev.preventDefault();
 const q = document.getElementById('q').value.trim();
 if (!q) return false;
 document.getElementById('b').disabled = true;
 userMsg(q); document.getElementById('q').value = '';
 const th = thinkingMsg();                       // live thinking feedback
 const t0 = Date.now();
 const secs = () => Math.round((Date.now() - t0) / 1000);
 let oldHits = 0;
 const poll = async () => { try {
  const r = await fetch('/progress?sid=' + encodeURIComponent(SID));
  if (r.status === 404) {                        // stale server without /progress
   if (++oldHits >= 3) tickThink(th,
    'old service instance (no live steps) — re-run the cells and open the new URL');
   return; }
  const p = await r.json();
  if (!p.n || p.done) { tickThink(th, 'thinking… ' + secs() + ' s'); return; }
  const s = p.steps[p.steps.length - 1];         // only THIS turn's steps
  tickThink(th, `step ${p.n} · ${s.kind}: ${s.text} · ${secs()} s`);
 } catch (e) {} };
 const timer = setInterval(poll, 700); poll();
 try { const r = await fetch('/chat', {method: 'POST',
  headers: {'Content-Type': 'application/json'},
  body: JSON.stringify({message: q, session_id: SID})});
  botMsg(await r.json());
 } catch (e) { botMsg({events: [], answer: 'service unreachable: ' + e}); }
 clearInterval(timer); th.remove();
 document.getElementById('b').disabled = false;
 window.scrollTo({top: document.body.scrollHeight, behavior: 'smooth'});
 return false; }
fetch('/schema').then(r => r.json()).then(s => {
 if (!s.llm_key_set) document.getElementById('banner').style.display = 'block';
 if (s.forecast_available === false)
  document.querySelector('.sub').textContent += ' · forecast disabled (no TCN bundle)'; });
chipRow(); document.getElementById('q').focus();
</script></body></html>"""
PAGE = PAGE.replace('__ISLANDS__', _ISL or 'supported conditions').replace(
    '__MODE__', 'fixture smoke-test run' if FIXTURE else 'real bundles loaded')

@app.get('/', response_class=HTMLResponse)
def index():
    return PAGE

# ---- end-to-end test of every routing row (falls back offline if needed) ------
from fastapi.testclient import TestClient

def chat_e2e(client, message, sid, force_offline=False):
    """POST /chat; if the real LLM is unreachable (or absent, or the check must be
    deterministic) the SAME loop is re-run with the offline placeholder so the
    route is still verified."""
    def _offline():
        _saved = call_llm
        globals()['call_llm'] = lambda m, t=None: fake_llm_turn(m, t)
        try:
            return client.post('/chat',
                               json={'message': message,
                                     'session_id': sid + '_off'}).json()
        finally:
            globals()['call_llm'] = _saved
    if force_offline:
        return _offline()
    d = client.post('/chat', json={'message': message, 'session_id': sid}).json()
    if d.get('answer', '').startswith('[LLM unavailable') or not LLM_API_KEY:
        d = _offline()
    return d

_c = TestClient(app)
_r0 = _c.get('/')
assert _r0.status_code == 200 and 'BoilDesc' in _r0.text, 'GET / failed'
_r1 = _c.get('/schema')
assert _r1.status_code == 200 and _r1.json()['islands'] and \
    _r1.json()['routing'], 'GET /schema failed'

# render route
_d = chat_e2e(_c, 'what does 21 degrees C look like?', 'e2e_render')
assert _d.get('renders'), \
    f'render route failed: {json.dumps(_d.get("events"))[:300]}'
_img = _d['renders'][0]['images']
from PIL import Image
for key in ('m4', 'm2'):
    p = os.path.join(RUNTIME_DIR, _img[key].lstrip('/'))
    im = Image.open(p).convert('L')
    hist = im.histogram()
    assert os.path.getsize(p) > 0 and (1 - hist[255] / sum(hist)) > 0.02, \
        f'{key} image looks blank'
    _a = np.asarray(im, dtype=float)             # nothing may touch the border:
    assert (_a[:4, :] < 200).sum() + (_a[-4:, :] < 200).sum() \
        + (_a[:, :3] < 200).sum() + (_a[:, -3:] < 200).sum() == 0, \
        f'{key} image is clipped at the figure edge (title overflow)'
assert _img['shared_scale'] is True and _img['m4_range'] == _img['m2_range'], \
    'heat maps must share ONE color scale (same shade = same temperature)'
assert _d.get('summary'), 'the reply summary (In short block) is missing'

# live progress endpoint (the browser thinking indicator's data source)
_pr = _c.get('/progress', params={'sid': 'e2e_render'})
assert _pr.status_code == 200 and _pr.json()['done'] and _pr.json()['n'] >= 3, \
    f'progress endpoint did not report the finished chat: {_pr.json()}'

# forecast route (origin 'end' -> tail fast path)
if FC_AVAILABLE:
    _df = chat_e2e(_c, 'what happens next at 21 °C?', 'e2e_forecast')
    assert _df.get('forecasts'), \
        f'forecast route failed: {json.dumps(_df.get("events"))[:300]}'
    _fp = os.path.join(RUNTIME_DIR, _df['forecasts'][0]['image'].lstrip('/'))
    _fa = np.asarray(Image.open(_fp).convert('L'), dtype=float)
    assert (_fa[:4, :] < 200).sum() + (_fa[-4:, :] < 200).sum() \
        + (_fa[:, :3] < 200).sum() + (_fa[:, -3:] < 200).sum() == 0, \
        'forecast chart is clipped at the figure edge (suptitle overflow)'
    assert _df.get('summary'), 'forecast reply summary missing'

assert 'In short' in PAGE and 'sumbox' in PAGE and 'object-fit:contain' in PAGE, \
    'the page must render the reply summary and full (never cropped) images'
assert '<div class="foot">' in PAGE and '<div class="chips" id="chips">' in PAGE \
    and 'position:sticky;bottom:0;z-index:5' in PAGE, \
    'the question buttons must sit in the sticky bar on top of the input field'

# conditions route
_dl = chat_e2e(_c, 'which cases are in the data?', 'e2e_list')
assert _dl.get('lists'), \
    f'conditions route failed: {json.dumps(_dl.get("events"))[:300]}'

# reliability route: direct model-card answer, no tool call (deterministic check -
# the real LLM is allowed its own judgement, the wiring is what we verify here)
_dr = chat_e2e(_c, 'how reliable is D32?', 'e2e_card', force_offline=True)
assert _dr.get('answer') and not (_dr.get('renders') or _dr.get('forecasts')
                                  or _dr.get('lists')), \
    f'reliability route should not call tools: {json.dumps(_dr.get("events"))[:300]}'

# sensitivity route: two renders in one round
_ds = chat_e2e(_c, 'what if the void fraction doubles?', 'e2e_sens')
assert _ds.get('renders'), \
    f'sensitivity route failed: {json.dumps(_ds.get("events"))[:300]}'

# direct tool endpoints (no LLM at all)
_r3 = _c.post('/tools/render', json={'ask': {'wallTemp': 25.0}})
assert _r3.status_code == 200 and _r3.json()['merged']['rows']
if FC_AVAILABLE:
    _r4 = _c.post('/tools/forecast', json={'wallTemp': 20.0, 'horizon': 24})
    assert _r4.status_code == 200 and _r4.json().get('ok')
print(f"[e2e] GET / 200 · /schema (islands + routing) ok · /progress (live "
      f"thinking steps, {_pr.json()['n']} reported) ok · render images full-frame "
      f"(no edge clipping) on one shared color scale · reply summary present · "
      f"sticky question bar (buttons above the field) · routes: render "
      f"({len(_d['events'])} steps, images non-blank) · "
      + (f"forecast ({len(_df['events'])} steps) · " if FC_AVAILABLE else '')
      + f"conditions ({len(_dl['events'])} steps) · reliability (direct card answer, "
      f"no tool) · sensitivity ({len(_ds.get('renders', []))} renders) · "
      f"/tools/render + /tools/forecast 200")
print('[ok] CELL 8 done — service live (final cell starts it)')


[e2e] GET / 200 · /schema (islands + routing) ok · /progress (live thinking steps, 10 reported) ok · render images full-frame (no edge clipping) on one shared color scale · reply summary present · sticky question bar (buttons above the field) · routes: render (10 steps, images non-blank) · forecast (9 steps) · conditions (7 steps) · reliability (direct card answer, no tool) · sensitivity (2 renders) · /tools/render + /tools/forecast 200
[ok] CELL 8 done — service live (final cell starts it)


## 9 · Self-check and launch

The checklist re-runs the critical guarantees as one block — artifacts discovered,
the retriever identity **invariant**, the anchor round-trip as a median of five
rows against the leaderboard reference (mean 11.7 %, median 10.9 %, p90 22.4 %;
a single blown anchor can no longer fail a good bundle), merge-table sanity,
finite TCN rollout, and the routing of the canonical questions — then starts
uvicorn **in a background daemon thread**: Jupyter already runs an event loop,
and `uvicorn.run()` would call `asyncio.run()` inside it (the classic
"cannot be called from a running event loop" RuntimeError). The thread owns its
own loop, the cell returns immediately (the kernel stays usable), a `/card`
probe confirms the server is really up, and the thread dies with the kernel —
no orphan process. In fixture mode no server is started.

The launch also handles the **stale-server trap** (the Windows "only one usage
of each socket address" 10048 error): re-running the notebook without
restarting the kernel used to leave the OLD server holding the port — the new
code never bound, and the browser kept talking to the stale instance (old
page, no live thinking steps) while the probe answered 200 from it. Now a
server started earlier **by this kernel is shut down first** (and the port
waited for), a port still held by an **older BoilDesc run or another program**
makes the launch **move to the next free port** with a loud note, and "up" is
only claimed when OUR server answers on OUR port — the printed URL is always
the one to open.

In [9]:
# CELL 9 · self-check + launch
#
# The per-cell smokes already asserted the pieces; this cell re-runs the critical
# ones as one checklist and then (outside fixture mode) starts the server.
# v2: same philosophy as CELL 3 — invariants instead of index coincidences, and
# thresholds generous enough that a single unlucky frame cannot fail a good
# bundle (the anchor check uses the MEDIAN of 5 spread rows vs the leaderboard
# reference, not one row against the mean).
checks = []

def check(name, fn):
    try:
        ok, detail = fn()
    except Exception as e:
        ok, detail = False, f'error: {e}'
    checks.append((name, bool(ok), str(detail)))

check('artifacts discovered',
      lambda: (bool(ART['m4'] and ART['m2'] and ART['h5']),
               f"{len(ART['h5'])} h5 file(s), M4 + M2"
               + (" + TCN" if ART['fc'] else " (no TCN)")))

def _m2_identity():
    """the true invariant: a stored row retrieves a numerically identical row
    (duplicates may tie; row 0 must retrieve itself - nothing precedes it)."""
    worst = 0.0
    for j in (0, len(M2COND) // 2, len(M2COND) - 1):
        k, d, _s, _p = retrieve(M2COND[j])
        worst = max(worst, float(np.linalg.norm(M2COND[k] - M2COND[j])))
    k0, d0, _, _ = retrieve(M2COND[0])
    ok = (k0 == 0 and d0 < 1e-5 and worst < 1e-5)
    return ok, (f'row 0 exact self-retrieval, rows [0, mid, last] nearest-gap '
                f'{worst:.2e} (duplicates are legitimate ties)')
check('M2 self-retrieval identity', _m2_identity)

def _m4_anchor():
    if FIXTURE:
        return True, 'fixture: random weights, RT not meaningful'
    _idx = np.linspace(0, len(M2COND) - 1, 5).astype(int)
    live = np.array([c in set(LIVE_COLS) for c in FRAME_COLS])
    sd = np.array([TH_SD[c] for c in FRAME_COLS])
    rts = []
    for j in _idx:
        z = M2COND[j:j + 1]
        d, T, _ = generate(z)
        th = apply_cal(theta_of_frames(d, T, [t_hot_of_cond_row(z[0])]))[0]
        ask = np.array([phys_of_z(z[0]).get(c, np.nan) for c in FRAME_COLS],
                       np.float64)   # .get: dead frame columns are not conditioned on
        rts.append(100 * float(np.mean(np.abs(th[live] - ask[live]) / sd[live])))
    med = float(np.median(rts))
    # generous bound: a good bundle medians ~11%; a blown single anchor must not
    # fail the run (the leaderboard p90 is 22.4%)
    return med < 40, (f'median of rows {_idx.tolist()} = {med:.1f}% '
                      f'(individual: {", ".join(f"{r:.1f}" for r in rts)}; '
                      f'reference: mean 11.7%, median 10.9%, p90 22.4%)')
check('M4 anchor round-trip', _m4_anchor)

def _merge_sanity():
    wt = float(np.mean(_wts_sb)) if _wts_sb else 20.0
    r = tool_boiling_render({'wallTemp': wt})
    rows = r['merged']['rows']
    ok = (len(rows) == len(LIVE_COLS)
          and all(rr['range_lo'] <= rr['range_hi'] for rr in rows)
          and all(rr['verdict'] in ('agree', 'partial', 'disputed') for rr in rows))
    return ok, r['merged']['summary']
check('merge table sanity', _merge_sanity)

if FC_AVAILABLE:
    def _fc_finite():
        f = tool_boiling_forecast(horizon=20)
        if not f.get('ok'):
            return False, f.get('reason', '')
        vals = [v for arr in f['trajectory'].values() for v in arr]
        return all(np.isfinite(vals)), (f"20 steps from {f['stats']['stem']}, "
                                        f"onset step {f['stats']['onset_step']}")
    check('TCN rollout finite', _fc_finite)

def _routing():
    """the documented routing must classify the canonical questions correctly."""
    _cases = [
        ('what does the wall region look like at 21 C?', 'render'),
        ('what happens next at 21 C?', 'forecast'),
        ('when does the first bubble appear?', 'forecast'),
        ('which cases are in the data?', 'conditions'),
        ('how reliable is D32?', 'reliability'),
        ('what is the heat flux?', 'not_available'),
        ('what if the void fraction doubles?', 'sensitivity'),
    ]
    _bad = [f'{m!r} -> {route_intent(m)["intent"]} (want {want})'
            for m, want in _cases if route_intent(m)['intent'] != want]
    return not _bad, f'{len(_cases) - len(_bad)}/{len(_cases)} canonical questions ' \
        + ('routed correctly' if not _bad else f'misrouted: {_bad}')
check('question routing', _routing)

_w = max(len(n) for n, _o, _d in checks)
_all = True
for name, ok, detail in checks:
    _all &= ok
    print(f"[{'PASS' if ok else 'FAIL'}] {name:<{_w}}  {detail}")
assert _all, 'self-check failed - see the FAIL lines above'
print(f"[ok] CELL 9 — all {len(checks)} self-checks PASS; service ready")

# ---- launch -------------------------------------------------------------------
# Jupyter already runs an event loop of its own, and uvicorn.run() calls
# asyncio.run() internally - that is exactly the "asyncio.run() cannot be
# called from a running event loop" RuntimeError. So the server runs in a
# background daemon thread (a fresh thread gets its own event loop): the cell
# returns immediately, the kernel stays usable, and the thread dies with the
# kernel.
# v3: the stale-server trap (the Windows 10048 bind error). Re-running this
# cell or the whole notebook in the same kernel used to KEEP the old server
# on the port: the new code never bound (WinError 10048), and the browser kept
# talking to the OLD instance - old page, no /progress route, NO thinking
# feedback - while the probe happily answered 200 from that stale process. Now:
#   1. a server started earlier by THIS kernel is shut down first (should_exit,
#      then wait for the port to actually be released);
#   2. if the port is still held by someone else (an older kernel still alive
#      in another tab, a leftover process, another program), the launch MOVES
#      to the next free port and says so loudly - open the printed URL, not the
#      stale one;
#   3. "up" is only claimed when OUR server answers the probe on OUR port.
if FIXTURE:
    print('[serve] FIXTURE mode — no server (synthetic placeholder data).')
    print(f'         On the target machine (DATA_DIR={DATA_DIR}) this cell serves '
          f'http://127.0.0.1:{PORT} in a background thread.')
else:
    import socket as _socket, threading, time, requests, uvicorn

    def _port_in_use(port):
        with _socket.socket(_socket.AF_INET, _socket.SOCK_STREAM) as s:
            s.settimeout(0.25)
            return s.connect_ex(('127.0.0.1', port)) == 0

    def _looks_like_boildesc(port):
        """is the port's occupant an older instance of THIS service?"""
        try:
            r = requests.get(f'http://127.0.0.1:{port}/card', timeout=2)
            return r.status_code == 200 and 'floors' in r.json()
        except Exception:
            return False

    # 1. stop a server this kernel started earlier (re-run of this cell/notebook)
    _prev = globals().get('_UVICORN_STATE')
    if _prev is not None:
        _ps, _pt, _pp = _prev
        if _pt.is_alive():
            _ps.should_exit = True
            for _ in range(60):                       # <= ~3 s for a clean stop
                if not _pt.is_alive():
                    break
                time.sleep(0.05)
            print(f'[serve] stopped the previous server started by this kernel '
                  f'(port {_pp} released).')
    # 2. pick the port: the configured one, or the next free one if it is held
    _port, _note = PORT, ''
    if _port_in_use(_port):
        if _looks_like_boildesc(_port):
            _note = (f'port {_port} is still held by an OLDER BoilDesc run (a '
                     'previous kernel that is still alive) - that old page has no '
                     'live thinking feedback; close that kernel to free the port')
        else:
            _note = f'port {_port} is used by another program'
        for _cand in range(PORT + 1, PORT + 6):
            if not _port_in_use(_cand):
                _port = _cand
                break
        print(f'[serve] {_note} - this run moves to port {_port}.')
    # 3. start, then confirm OUR server answers on OUR port before claiming "up"
    _srv = uvicorn.Server(uvicorn.Config(app, host='127.0.0.1', port=_port,
                                         log_level='warning'))
    _thr = threading.Thread(target=_srv.run, name='boildesc-uvicorn', daemon=True)
    _thr.start()
    _url = f'http://127.0.0.1:{_port}'
    _up = False
    for _ in range(60):                               # <= ~6 s startup window
        time.sleep(0.1)
        try:
            if requests.get(f'{_url}/card', timeout=2).status_code == 200:
                _up = True
                break
        except Exception:
            pass
    _UVICORN_STATE = (_srv, _thr, _port)              # for the next re-run
    if _up:
        print(f'[serve] up on {_url} — open THIS url in the browser (its page polls '
              '/progress and shows the live thinking steps);\n'
              '         re-run this cell to restart the server here; restart the '
              'kernel to stop it.')
    else:
        print(f'[serve] FAILED to come up on {_url} (server thread alive: '
              f'{_thr.is_alive()}) - check the uvicorn message above, or set PORT '
              'in .env to another value and re-run this cell.')


[PASS] artifacts discovered        15 h5 file(s), M4 + M2 + TCN
[PASS] M2 self-retrieval identity  row 0 exact self-retrieval, rows [0, mid, last] nearest-gap 0.00e+00 (duplicates are legitimate ties)
[PASS] M4 anchor round-trip        median of rows [0, 4001, 8003, 12005, 16007] = 7.6% (individual: 7.6, 7.6, 6.5, 6.6, 13.9; reference: mean 11.7%, median 10.9%, p90 22.4%)
[PASS] merge table sanity          13/13 columns agree (<0.5 sigma), 0 disputed (>=1.5 sigma)
[PASS] TCN rollout finite          20 steps from Twall_17, onset step 1
[PASS] question routing            7/7 canonical questions routed correctly
[ok] CELL 9 — all 6 self-checks PASS; service ready
[serve] port 8000 is still held by an OLDER BoilDesc run (a previous kernel that is still alive) - that old page has no live thinking feedback; close that kernel to free the port - this run moves to port 8002.
[serve] up on http://127.0.0.1:8002 — open THIS url in the browser (its page polls /progress and shows the live thinking 

## 10 · Running it on the target machine

1. Put this notebook in a folder (its runtime artifacts — `.env`, images, audit log,
   fixtures — are created next to it).
2. `pip install torch h5py numpy scipy pandas matplotlib fastapi uvicorn requests
   pillow` once.
3. Make sure `G:\BoilDesc` (or set `DATA_DIR=` in `.env`) holds the 15 `.h5` files
   and `models\` with the three bundles.
4. Paste the Gemini key into `.env` → `LLM_API_KEY=...`.
5. Run all cells; the last one prints the URL to open (`http://127.0.0.1:8000`,
   or the next free port with a note if 8000 is still held by an older run —
   **open the printed URL**, and close the old kernel to free 8000).

Troubleshooting: *port busy / 10048* → an older BoilDesc kernel is still alive:
the launch moves to a free port and prints which — close that kernel (or set
`PORT` in `.env`) and re-run the last cell; *chat answers say "offline
placeholder"* → the key was not read, check `.env` spelling, or the API call
failed (the answer then carries the error); *no thinking steps under the
spinner* → the browser is on a stale instance from before the update —
re-run the cells and open the freshly printed URL; *"what happens next" is
fast but
"when does the first bubble appear" takes ~a minute once* → that question streams
the whole θ table of that file (progress is printed; cached afterwards);
*no TCN* → the `FC_F2_TCN_bundle_*` file was not found under `DATA_DIR` (render
still works). The key stays in `.env` next to the notebook and is never printed
or logged; the server binds to `127.0.0.1` only — change the host in the last
cell explicitly if you ever want to expose it.